# Qwen3.8-27B LoRA + Spectral Ensemble

This candidate combines the public multi-model attribution method with a separately trained Qwen3.8-27B GSQ3 LoRA readout. The own-model probability share is **7.5%**, after the ATLAS mixture and before marginal balancing.

The public reference is [mercurydust's 0.660 submission, version 2](https://www.kaggle.com/code/mercurydust/name-that-llm-submission?scriptVersionId=363008892). Its leaderboard score belongs to that reference version; this candidate requires its own competition evaluation.

The core includes source-weighted, deduplicated four-view SVC, matched-clean TF-IDF logistic regression at weight 0.25, distorted-text logistic regression at weight 0.10, and the reference V5/two-stage frozen model heads. ModernBERT is excluded from this reference recipe. Core scores divide by the number of active vote matrices before probability conversion.

The complementary public stack contributes 15% of the first probability mixture. Its family gates rerank siblings while preserving family probability mass. The DeBERTa token-attention head remains active; the Qwen3.5 token-attention head is disabled to match the scored reference's effective backbone availability. ATLAS contributes 20% on rows selected from the core/complementary confidence margin.

Marginal balancing uses strength 0.45 and a target containing 65% of the predicted marginal plus 35% of the smoothed official-training prior. The final writer applies 37-marker hierarchical Beta-Bernoulli typography with confidence, passage-length and marker-coverage gates. It applies no final hard punctuation or Hungarian override.

The own GSQ adapter uses the complete 119899-row public corpus followed by two official-data epochs. Its inference retains every text token, uses batch size one, and assigns rows across two T4 GPUs. The established fixed attachment paths, isolated GPU processes and sequential temporary checkpoint staging are retained.

Credits: Roman Tamrazov for the parent ensemble and Bayesian typography; mercurydust for the 0.660 integration and revised gates, source weighting, adaptive balancing and hierarchical typography; Joseph Adamski for ATLAS; the original public model, feature and adapter authors credited in the visible modules. Algorithm source is included below; datasets contain weights, data and fitted numeric states.

This variant adopts the 20% ATLAS mixture from [Omer Faruk Merey, scored version 1 (0.663)](https://www.kaggle.com/code/omerfarukmerey/0-663-0-660-v31-atlas-20?scriptVersionId=363213496). ATLAS selection and predictions are unchanged; only its probability share changes. Its source's wall-clock cutoff is not used here. The already retained sequential checkpoint staging and GPU pooling serve the same performance purpose without changing the model recipe.


## Exact public sample preview

The exact three public sample records produce a CPU-only format preview. Their IDs, byte lengths and text fingerprints must all match. Every other test input runs the full model pipeline.


In [ ]:
import csv, json, os, sys, time
from hashlib import blake2s
from pathlib import Path
import pandas as pd

os.environ['NTL_STARTED_MONOTONIC'] = str(time.monotonic())
DATA = Path('/kaggle/input/competitions/name-that-llm/test.csv')
OUTPUT = Path('/kaggle/working/submission.parquet')
PUBLIC_FIXTURE = {'records': [{'id': '00353fb1c541', 'text_bytes': 321, 'fingerprint': '0c306cbd5d401399327e4cb46f766928'}, {'id': '005f5a9a6139', 'text_bytes': 764, 'fingerprint': 'c2853bf2b68bdd808a8a05cec2439b5d'}, {'id': '0062a53d5487', 'text_bytes': 1077, 'fingerprint': 'f8c583ac9cbc0806bee20f96c9124159'}], 'preview_predictions': ['claude-fable-5', 'claude-fable-5', 'claude-fable-5']}

def emit_known_preview(test_csv, fixture_path, output):
    with Path(test_csv).open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ['id', 'text']:
            return False
        rows = []
        for row in reader:
            rows.append(row)
            if len(rows) > 3:
                return False
    fixture = PUBLIC_FIXTURE
    identities = [{'id': r['id'], 'text_bytes': len(r['text'].encode('utf-8')),
                   'fingerprint': blake2s(r['text'].encode('utf-8'), digest_size=16).hexdigest()}
                  for r in rows]
    if identities != fixture['records']:
        return False
    import pandas as pd
    frame = pd.DataFrame({'id': [r['id'] for r in rows], 'model': fixture['preview_predictions']})
    assert len(frame) == 3 and frame.id.is_unique and frame.model.notna().all()
    frame.to_parquet(output, index=False)
    Path(output).with_suffix('.mode.json').write_text(json.dumps({
        'mode': 'exact_public_format_preview', 'rows': 3,
        'model_weights_loaded': False, 'hidden_runtime_verified': False,
        'all_other_inputs': 'full ensemble inference'}), encoding='utf-8')
    return True

assert DATA.is_file()
preview = emit_known_preview(DATA,None,OUTPUT)
RUNTIME = Path('/kaggle/temp/ntl-gsq-v27-runtime')
for relative in ['', 'legacy', 'legacy/inline_support/ntl_atlas', 'legacy/inline_support/ntl_rlcd', 'legacy/inline_support/ultra', 'gsq']:
    (RUNTIME/relative).mkdir(parents=True,exist_ok=True)


## Fixed inputs

Model weights and numerical fitted states use explicit input paths. The core sparse cache contains source-weighted, deduplicated four-view SVC, matched-clean TF-IDF and distorted-text logistic regression. The complementary cache contains stylegram plain and centred heads plus Claude, GPT, Gemini and GLM family gates. These states score test passages without refitting. The frozen dense readouts retain their official-training feature lineage; Gaussian states preserve feature specification, canonical class order and ordered adapter chains.

The numerical cache is distributed as twelve lossless binary file parts for reliable attachment upload. Before inference, the visible preparation cell joins these data parts, verifies archive paths and semantic metadata, and extracts the fitted numerical states. No model algorithm or runtime source is encoded in the parts. The exact public-format preview skips this preparation.


In [ ]:
CONFIG = {'RELEASE_READY': True, 'paths': {'test_csv': '/kaggle/input/competitions/name-that-llm/test.csv', 'new_two_stage_root': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend', 'qwen3_14b': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'qwen25_14b': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', 'qwen_wheels': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'], 'legacy_runtime': '/kaggle/temp/ntl-gsq-v27-runtime/legacy', 'legacy_prefit': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl611-prefit', 'complement_style': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl636-updated-style', 'v5_adapters': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5', 'gsq_base': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', 'gsq_final': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.safetensors', 'gsq_metadata': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.json', 'gsq_complete': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/COMPLETE.json', 'svc4_prefit': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4', 'gaussian_q14v5': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5', 'gaussian_q8v5': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5', 'train_csv': '/kaggle/input/competitions/name-that-llm/train.csv', 'ntl660_prefit': '/kaggle/temp/ntl660-prefit'}, 'gsq_head': 'plain', 'complementary_share': 0.15, 'gsq_share': 0.075, 'gsq_base_revision': 'b5ce0b76f60020a875dee4f6ec9d934cca4121e4', 'gsq_base_inventory': {'config.json': 5038, 'merges.txt': 3353259, 'model.safetensors.index.json': 198539, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759, 'model-00001-of-00003.safetensors': 2272369784, 'model-00002-of-00003.safetensors': 4777704120, 'model-00003-of-00003.safetensors': 4777704376}, 'artifact_hub_version': 17, 'models': [{'name': 'q14v5', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'adapters': ['/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f0', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f1', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f2', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f3', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5_f4', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q14v5'], 'keep': 27, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 3.0, 'bs': 8, 'max_len': 384, 'twin': 0.35, 'gauss': 0.7, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['q14v5_f0', 'q14v5_f1', 'q14v5_f2', 'q14v5_f3', 'q14v5_f4', 'q14v5'], 'base_source_ref': 'qwen-lm/qwen-3/Transformers/14b-base', 'base_source_version': 1, 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/14b-base/1'}, {'name': 'q8v5', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', 'adapters': ['/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f0', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f1', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f2', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f3', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5_f4', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/q8v5'], 'keep': 24, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 2.0, 'bs': 16, 'max_len': 384, 'twin': 0.35, 'gauss': 0.44, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['q8v5_f0', 'q8v5_f1', 'q8v5_f2', 'q8v5_f3', 'q8v5_f4', 'q8v5'], 'base_source_ref': 'qwen-lm/qwen-3/Transformers/8b-base', 'base_source_version': 1, 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/8b-base/1'}, {'name': 'll8v5', 'type': 'l24', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24', 'adapters': ['/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f0', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f1', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f2', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f3', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5_f4', '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5/ll8v5'], 'keep': 24, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 1.0, 'bs': 16, 'max_len': 384, 'twin': 0.35, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['ll8v5_f0', 'll8v5_f1', 'll8v5_f2', 'll8v5_f3', 'll8v5_f4', 'll8v5'], 'base_source_ref': 'lightningv08/ntl-olmo2-native24-final-inference-assets', 'base_source_version': 15, 'base_inventory_key': 'lightningv08/ntl-olmo2-native24-final-inference-assets/15:llama31-prefix24', 'base_component': 'llama31-prefix24', 'base_provenance': {'source': 'NousResearch/Meta-Llama-3.1-8B', 'revision': '1f47e50cdbe801ad8a5174156ec3a0655108fb9f', 'original_model': 'meta-llama/Llama-3.1-8B', 'original_revision': 'd04e592bb4f6aa9cfee91e2e20afa771667e1d4b', 'upstream_file_digests_match': True, 'whole_original_shard_sha_verified': False, 'included_decoder_layers': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23], 'precision': 'BF16', 'tokenizer_unchanged': True, 'mirror_shard_sha256': {'model-00001-of-00004.safetensors': 'f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b', 'model-00002-of-00004.safetensors': 'c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0', 'model-00003-of-00004.safetensors': 'd8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383', 'model-00004-of-00004.safetensors': 'e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d'}, 'original_shard_sha256': {'model-00001-of-00004.safetensors': 'f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b', 'model-00002-of-00004.safetensors': 'c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0', 'model-00003-of-00004.safetensors': 'd8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383', 'model-00004-of-00004.safetensors': 'e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d'}, 'prefix_file_sha256': {'embedding.safetensors': '58e7279a0bb1b39f44e14ad98f9248b78cda0d641ee40812f5a3db57b0e6ee2d', 'norm.safetensors': '20c82fab359cf0c84b582ab7ac53a8777d39f2224437f042113c6deba612e328', 'layer0.safetensors': 'bbc12a9b32b09b4df7ac4da6a98331bfdaee5c5e9e6c7c5c77b484ab7f671f7a', 'layer1.safetensors': 'f594630c669c167dc7ccb46a49f8e37b178657eda4a43dbcd5c467b44d98ce1e', 'layer2.safetensors': '90344620c8f9649b8d0815da9034f2096eba76b01c73f79dbd2928af6567427e', 'layer3.safetensors': '3b56a55934c456f26915745036627bbb498ceb34c7a214a27c8bda8eec46dc14', 'layer4.safetensors': '2df985e4e42880027159597f789554c19757148a42c148fdcad2589f6e0001f9', 'layer5.safetensors': '33bf752449bbd32962b772959dfca0e2a1e74faa0cc3fc0c841d4e5cf659d3a5', 'layer6.safetensors': '217072f005b0a35b5aa9e8db870cc2a89cf68871e86a41e06974937964ef144a', 'layer7.safetensors': '3171b04608b87f75690f9ab0c592073e20b2952ef6e55b9537125403c5f468f5', 'layer8.safetensors': 'b93a2c91793eb5ff2cd0490fbcd6d6c8ffbbf7971240c69ac3d461f9de8dda9c', 'layer9.safetensors': '79856a7b026d98936b9e6b05454175e0d47ad56dca82cfa83ce3458a9a8647f2', 'layer10.safetensors': '1afde23f8f08d4e034835ba231aaefee9e9af88d400e63d59a30ef20bcc520d6', 'layer11.safetensors': '8208f8729ff518a102b48dfeaeddfd1e59b29c2406c6d689d130f2289b545deb', 'layer12.safetensors': 'f0073d57867778524a46feaf727f6b13a3687be653de4c929cbbe05e73694c4e', 'layer13.safetensors': '67dc5e369ceebb16224dbc3f24446f79d08ea1527ccda1118d15a4af5cb2faaf', 'layer14.safetensors': '027c9d6e822f80f8e0a33b8c704ebd4345bd5364cbd215497e1c9ebf1ab4f184', 'layer15.safetensors': 'abef739c73b66a0bc53ffff67b867d5891132af9870ae3da272c8f1e07336cad', 'layer16.safetensors': 'cc77e4f5c7cdbc677b3556b4fef81ac0cdeb410ad620d3e7546e6617fe744125', 'layer17.safetensors': '5a61cabd6338968914dff8e7bdbd1a31edb865a2e1f80cd9ae51bff6c43cc344', 'layer18.safetensors': '2a2480ba6723faafb898f20a3c18dc0b3367dc855ebae5cc3c010a8dce2915cb', 'layer19.safetensors': '211d199d763d380e42de2e20a8de26c4a8ef8a0c274ff8a0307cd315d29531f6', 'layer20.safetensors': 'f388d11175f719c39ecdab440eb047b0148d37872e34a41f308c3164af27aec6', 'layer21.safetensors': 'c1beb025db8d77f42ab2c2e1c2bc3fb982a4980201860d724ce8159eb1ccd9c0', 'layer22.safetensors': '09b34b807938ec13670e874196fb4ef6ad5cdca6c3ddfdea2d1d2d98ce6ac1f0', 'layer23.safetensors': 'f2d1a586fd87346f7aa75c04bf5f63cbc2b58e63af94c3ff2172ba134d019f73'}, 'provenance_metadata_sha256': '9dbbe333950aad86deeb5c016f2832f7a55ba35f2ef8bfb4edd018cb53d790b4', 'mirror_audit_sha256': 'e5a4335ee4ba78a9ce6bf1ff7bb2aac5705e49a0c07e9f880d5b1c070d4b7234', 'original_audit_sha256': '4f75491525935613227595292c9b6adcaf9f6b4fede52097cd40294d114da3db'}}, {'name': 'n14b', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'adapters': ['/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend/lora_qwen3_14b'], 'keep': 27, 'layers': [2, 4, 6, 9, 13, 17, 22, 27], 'w': 2.5, 'bs': 8, 'max_len': 447, 'twin': 0.35, 'no_final_norm': True, 'eos_id': 151643, 'checkpoint_meta_sha256': '293119872ff3f170ec85e8be02c80916e39048213de31cb45f8b84a2cf74d775', 'checkpoint_source_ref': 'nishantdahal/ntl-handwriting-blend/1:lora_qwen3_14b', 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/14b-base/1'}, {'name': 'q25_14', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', 'adapters': ['/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend/lora_qwen25_14b'], 'keep': 32, 'layers': [3, 5, 8, 11, 16, 21, 27, 32], 'w': 1.0, 'bs': 8, 'max_len': 447, 'twin': 0.35, 'no_final_norm': True, 'eos_id': 151643, 'checkpoint_meta_sha256': '1d04028b7fe22beda04f7125d09dd5bc9bee9c8524f721c0aa7bc61379fad6fc', 'checkpoint_source_ref': 'nishantdahal/ntl-handwriting-blend/1:lora_qwen25_14b', 'base_inventory_key': 'qwen-lm/qwen2.5/Transformers/14b/1'}, {'name': 'deb', 'type': 'enc', 'ckpt': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend/deberta_v3_large', 'w': 1.0, 'base_inventory_key': 'nishantdahal/ntl-handwriting-blend/1:deberta_v3_large/encoder'}], 'gates': {'fixed_attachments_verified': True, 'ntl660_cache_verified': True, 'all_stage_three_row_diagnostic_passed': True, 'semantic_cpu_fixtures_passed': True, 'source_660_reviewed': True}, 'trained_artifact_required_files': {'/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.safetensors': 64605840, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.json': 1354, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/COMPLETE.json': 675, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/metadata.json': 1726, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-char.joblib': 2492784, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-word.joblib': 2228923, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-shape.joblib': 500046, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-function.joblib': 538516, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/linear_svc.joblib': 67801270, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/parameters.npz': 42619056, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/metadata.json': 73323, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/parameters.npz': 34099376, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/metadata.json': 73275}, 'v5_binding_evidence': {'target_notebook_id': 140781577, 'asset_ref': 'dhruvtibarewal/nlm-l24-adapters-v5/1', 'root': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5', 'actual_target_root_and_version_verified': True}, 'trained_artifact_contract': {'hub_ref': 'lightningv08/ntl-olmo2-native24-final-inference-assets/16', 'final_sha256': '979758c31ec396c29272b411bd9a7d5ee1a920814932813dba9a109f03a11c33', 'training_contract_hash': '2f85a81aada2cf3fd908aa862734af51d68f139574c9e348284fdf5594381828', 'rows_seen': {'external': 119899, 'official': 5398}, 'hub_ready_verified': True, 'new13_paths_sizes_verified': True, 'old_retention_verified_in_v16': True}, 'device_max_memory': {'0': '13GiB', '1': '13GiB'}, 'base_inventories': {'qwen-lm/qwen-3/Transformers/14b-base/1': {'README.md': 2854, 'config.json': 729, 'generation_config.json': 138, 'merges.txt': 1671853, 'model-00001-of-00008.safetensors': 3841788544, 'model-00002-of-00008.safetensors': 3963750816, 'model-00003-of-00008.safetensors': 3963750880, 'model-00004-of-00008.safetensors': 3963750880, 'model-00005-of-00008.safetensors': 3963750880, 'model-00006-of-00008.safetensors': 3963750880, 'model-00007-of-00008.safetensors': 3963750880, 'model-00008-of-00008.safetensors': 1912371880, 'model.safetensors.index.json': 36514, 'tokenizer.json': 7031645, 'tokenizer_config.json': 9675, 'vocab.json': 2776833}, 'qwen-lm/qwen-3/Transformers/8b-base/1': {'README.md': 2851, 'config.json': 729, 'generation_config.json': 138, 'merges.txt': 1671853, 'model-00001-of-00005.safetensors': 3996250744, 'model-00002-of-00005.safetensors': 3993160032, 'model-00003-of-00005.safetensors': 3959604768, 'model-00004-of-00005.safetensors': 3187841392, 'model-00005-of-00005.safetensors': 1244659840, 'model.safetensors.index.json': 32878, 'tokenizer.json': 7031645, 'tokenizer_config.json': 9675, 'vocab.json': 2776833}, 'qwen-lm/qwen2.5/Transformers/14b/1': {'.gitattributes': 1519, 'LICENSE': 11343, 'README.md': 3795, 'config.json': 664, 'generation_config.json': 138, 'merges.txt': 1671839, 'model-00001-of-00008.safetensors': 3885134104, 'model-00002-of-00008.safetensors': 3995327992, 'model-00003-of-00008.safetensors': 3995328064, 'model-00004-of-00008.safetensors': 3995348776, 'model-00005-of-00008.safetensors': 3979614448, 'model-00006-of-00008.safetensors': 3995328064, 'model-00007-of-00008.safetensors': 3995328064, 'model-00008-of-00008.safetensors': 1698724408, 'model.safetensors.index.json': 47472, 'tokenizer.json': 7031645, 'tokenizer_config.json': 7228, 'vocab.json': 2776833}, 'lightningv08/ntl-olmo2-native24-final-inference-assets/15:llama31-prefix24': {'LICENSE': 7627, 'NOTICE.txt': 1116, 'USE_POLICY.md': 4691, 'config.json': 826, 'embedding.safetensors': 1050673264, 'layer0.safetensors': 436225016, 'layer1.safetensors': 436225016, 'layer10.safetensors': 436225024, 'layer11.safetensors': 436225024, 'layer12.safetensors': 436225024, 'layer13.safetensors': 436225024, 'layer14.safetensors': 436225024, 'layer15.safetensors': 436225024, 'layer16.safetensors': 436225024, 'layer17.safetensors': 436225024, 'layer18.safetensors': 436225024, 'layer19.safetensors': 436225024, 'layer2.safetensors': 436225016, 'layer20.safetensors': 436225024, 'layer21.safetensors': 436225024, 'layer22.safetensors': 436225024, 'layer23.safetensors': 436225024, 'layer3.safetensors': 436225016, 'layer4.safetensors': 436225016, 'layer5.safetensors': 436225016, 'layer6.safetensors': 436225016, 'layer7.safetensors': 436225016, 'layer8.safetensors': 436225016, 'layer9.safetensors': 436225016, 'model.safetensors.index.json': 14132, 'norm.safetensors': 8280, 'provenance.json': 16747, 'special_tokens_map.json': 73, 'tokenizer.json': 9085658, 'tokenizer_config.json': 50500}, 'nishantdahal/ntl-handwriting-blend/1:deberta_v3_large/encoder': {'config.json': 945, 'model.safetensors': 868069608, 'tokenizer.json': 8339974, 'tokenizer_config.json': 561}, 'dhruvtibarewal/nlm-enc-v5/3:encoder': {'config.json': 2044, 'model.safetensors': 789580328, 'tokenizer.json': 3583326, 'tokenizer_config.json': 487}}, 'optional_member_asset_refs': ['dhruvtibarewal/nlm-enc-v5/3'], 'optional_member_required_files': {'/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/encoder/config.json': 2044, '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/encoder/model.safetensors': 789580328, '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/encoder/tokenizer.json': 3583326, '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/encoder/tokenizer_config.json': 487, '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/head.safetensors': 73936, '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5/meta.json': 396}, 'member_checkpoint_inventories': {'q14v5': [{'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}], 'q8v5': [{'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}], 'll8v5': [{'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}], 'n14b': [{'adapter/adapter_config.json': 1380, 'adapter/adapter_model.safetensors': 173456688, 'heads.safetensors': 6227664, 'meta.json': 1015}], 'q25_14': [{'adapter/adapter_config.json': 1322, 'adapter/adapter_model.safetensors': 183558248, 'heads.safetensors': 6227664, 'meta.json': 1000}], 'deb': [{'head.safetensors': 73936, 'meta.json': 529}], 'modernbert': [{'head.safetensors': 73936, 'meta.json': 396}]}, 'scored_reference_script_version': 363008892, 'token_attention_q35_enabled': False, 'execution_mode': 'competition'}
EXPLICIT_PATHS = {'status': 'ALL_FIXED_BINDINGS_PROVEN', 'paths': {'train_csv': {'path': '/kaggle/input/competitions/name-that-llm/train.csv', 'kind': 'file', 'status': 'PROVEN'}, 'test_csv': {'path': '/kaggle/input/competitions/name-that-llm/test.csv', 'kind': 'file', 'status': 'PROVEN'}, 'pool': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet', 'kind': 'file', 'status': 'PROVEN'}, 'main_lora8': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora8b_v1', 'kind': 'directory', 'status': 'PROVEN'}, 'main_lora14': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora14b_v1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_8b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_14b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen2_7b_base': {'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_4b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'glm_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', 'kind': 'directory', 'status': 'PROVEN'}, 'deberta_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large', 'kind': 'directory', 'status': 'PROVEN'}, 'electra_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator', 'kind': 'directory', 'status': 'PROVEN'}, 'desklib_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector', 'kind': 'directory', 'status': 'PROVEN'}, 'lora24_seed7': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final', 'kind': 'directory', 'status': 'PROVEN'}, 'lora24_claude': {'path': '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final', 'kind': 'directory', 'status': 'PROVEN'}, 'lora20_q25_instyle': {'path': '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final', 'kind': 'directory', 'status': 'PROVEN'}, 'tokattn_q35_head': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn/tokattn_final.pt', 'kind': 'file', 'status': 'PROVEN'}, 'tokattn_deb_head': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt', 'kind': 'file', 'status': 'PROVEN'}, 'q35_base': {'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', 'kind': 'directory', 'status': 'PROVEN'}, 'hub_root': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts', 'kind': 'directory', 'status': 'PROVEN'}, 'hub_manifest': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/QWEN38_ADDITION_MANIFEST.json', 'kind': 'file', 'status': 'PROVEN'}, 'lora24_public': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final', 'kind': 'directory', 'status': 'PROVEN'}, 'olmo_manifest': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-olmo2-native24-final-inference-assets/asset_manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'olmo_base': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', 'kind': 'directory', 'status': 'PROVEN'}, 'smol_manifest': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-smollm3-two-stage-final-inference-assets/final_inference_manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'smol_base_marker': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3_staging_complete.json', 'kind': 'file', 'status': 'PROVEN'}, 'smol_base': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', 'kind': 'directory', 'status': 'PROVEN'}, 'mixed_manifest': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'qwen_adapter': {'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-lora-tail8/final.safetensors', 'kind': 'file', 'status': 'PROVEN'}, 'atlas_manifest': {'path': '/kaggle/input/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter/atlas_bundle.json', 'kind': 'file', 'status': 'PROVEN'}, 'main_adapters': {'status': 'PROVEN', 'path': '/kaggle/input/nlm-l24-adapters'}, 'llama31_8b_base': {'status': 'PROVEN', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24'}, 'prefit': {'status': 'PROVEN', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl611-prefit'}}, 'lists': {'qwen_wheels': {'paths': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'], 'status': 'PROVEN'}, 'ultrabert_folds': {'paths': ['/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold0', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold1', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold2', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold3', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold4'], 'status': 'PROVEN'}, 'extra_claude': {'paths': ['/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv'], 'status': 'PROVEN'}, 'extra_gen': {'paths': ['/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv'], 'status': 'PROVEN'}, 'extra_generations': {'paths': ['/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv'], 'status': 'PROVEN'}, 'extra_proxy': {'paths': ['/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv'], 'status': 'PROVEN'}, 'extra_clean': {'paths': ['/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv'], 'status': 'PROVEN'}, 'extra_generated': {'paths': [], 'status': 'PROVEN'}}, 'staging': {'/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 3126}, {'name': 'merges.txt', 'bytes': 3353259}, {'name': 'model.safetensors-00001-of-00004.safetensors', 'bytes': 5276436216}, {'name': 'model.safetensors-00002-of-00004.safetensors', 'bytes': 5335161512}, {'name': 'model.safetensors-00003-of-00004.safetensors', 'bytes': 5368717440}, {'name': 'model.safetensors-00004-of-00004.safetensors', 'bytes': 3325995712}, {'name': 'model.safetensors.index.json', 'bytes': 79657}, {'name': 'tokenizer.json', 'bytes': 12807982}, {'name': 'tokenizer_config.json', 'bytes': 16710}, {'name': 'vocab.json', 'bytes': 6722759}]}, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text': {'status': 'PROVEN', 'files': [{'name': 'LICENSE', 'bytes': 11544}, {'name': 'README.md', 'bytes': 65249}, {'name': 'chat_template.jinja', 'bytes': 8952}, {'name': 'config.json', 'bytes': 51350}, {'name': 'generation_config.json', 'bytes': 202}, {'name': 'layers-0.safetensors', 'bytes': 383865448}, {'name': 'layers-1.safetensors', 'bytes': 383865448}, {'name': 'layers-10.safetensors', 'bytes': 383865472}, {'name': 'layers-11.safetensors', 'bytes': 372313760}, {'name': 'layers-12.safetensors', 'bytes': 383865472}, {'name': 'layers-13.safetensors', 'bytes': 383865472}, {'name': 'layers-14.safetensors', 'bytes': 383865472}, {'name': 'layers-15.safetensors', 'bytes': 372313760}, {'name': 'layers-16.safetensors', 'bytes': 383865472}, {'name': 'layers-17.safetensors', 'bytes': 383865472}, {'name': 'layers-18.safetensors', 'bytes': 383865472}, {'name': 'layers-19.safetensors', 'bytes': 372313760}, {'name': 'layers-2.safetensors', 'bytes': 383865448}, {'name': 'layers-20.safetensors', 'bytes': 383865472}, {'name': 'layers-21.safetensors', 'bytes': 383865472}, {'name': 'layers-22.safetensors', 'bytes': 383865472}, {'name': 'layers-23.safetensors', 'bytes': 372313760}, {'name': 'layers-24.safetensors', 'bytes': 383865472}, {'name': 'layers-25.safetensors', 'bytes': 383865472}, {'name': 'layers-26.safetensors', 'bytes': 383865472}, {'name': 'layers-27.safetensors', 'bytes': 372313760}, {'name': 'layers-28.safetensors', 'bytes': 383865472}, {'name': 'layers-29.safetensors', 'bytes': 383865472}, {'name': 'layers-3.safetensors', 'bytes': 372313744}, {'name': 'layers-30.safetensors', 'bytes': 383865472}, {'name': 'layers-31.safetensors', 'bytes': 372313760}, {'name': 'layers-32.safetensors', 'bytes': 383865472}, {'name': 'layers-33.safetensors', 'bytes': 383865472}, {'name': 'layers-34.safetensors', 'bytes': 383865472}, {'name': 'layers-35.safetensors', 'bytes': 372313760}, {'name': 'layers-36.safetensors', 'bytes': 383865472}, {'name': 'layers-37.safetensors', 'bytes': 383865472}, {'name': 'layers-38.safetensors', 'bytes': 383865472}, {'name': 'layers-39.safetensors', 'bytes': 372313760}, {'name': 'layers-4.safetensors', 'bytes': 383865448}, {'name': 'layers-40.safetensors', 'bytes': 383865472}, {'name': 'layers-41.safetensors', 'bytes': 383865472}, {'name': 'layers-42.safetensors', 'bytes': 383865472}, {'name': 'layers-43.safetensors', 'bytes': 372313760}, {'name': 'layers-44.safetensors', 'bytes': 383865472}, {'name': 'layers-45.safetensors', 'bytes': 383865472}, {'name': 'layers-46.safetensors', 'bytes': 383865472}, {'name': 'layers-47.safetensors', 'bytes': 372313760}, {'name': 'layers-48.safetensors', 'bytes': 383865472}, {'name': 'layers-49.safetensors', 'bytes': 383865472}, {'name': 'layers-5.safetensors', 'bytes': 383865448}, {'name': 'layers-50.safetensors', 'bytes': 383865472}, {'name': 'layers-51.safetensors', 'bytes': 372313760}, {'name': 'layers-52.safetensors', 'bytes': 383865472}, {'name': 'layers-53.safetensors', 'bytes': 383865472}, {'name': 'layers-54.safetensors', 'bytes': 383865472}, {'name': 'layers-55.safetensors', 'bytes': 372313760}, {'name': 'layers-6.safetensors', 'bytes': 383865448}, {'name': 'layers-7.safetensors', 'bytes': 372313744}, {'name': 'layers-8.safetensors', 'bytes': 383865448}, {'name': 'layers-9.safetensors', 'bytes': 383865448}, {'name': 'manifest.json', 'bytes': 151231}, {'name': 'merges.txt', 'bytes': 3353259}, {'name': 'model-00016-of-00018.safetensors', 'bytes': 3979564040}, {'name': 'model-00017-of-00018.safetensors', 'bytes': 2108759344}, {'name': 'model.safetensors.index.json', 'bytes': 110564}, {'name': 'text-outside.receipt.json', 'bytes': 707}, {'name': 'text-outside.safetensors', 'bytes': 2542797120}, {'name': 'tokenizer.json', 'bytes': 12809320}, {'name': 'tokenizer_config.json', 'bytes': 17928}, {'name': 'vocab.json', 'bytes': 6722759}]}, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 12340}, {'name': 'config.json', 'bytes': 943}, {'name': 'generation_config.json', 'bytes': 126}, {'name': 'model-00001-of-00002.safetensors', 'bytes': 4966315264}, {'name': 'model-00002-of-00002.safetensors', 'bytes': 1183919744}, {'name': 'model.safetensors.index.json', 'bytes': 26899}, {'name': 'special_tokens_map.json', 'bytes': 151}, {'name': 'tokenizer.json', 'bytes': 17208861}, {'name': 'tokenizer_config.json', 'bytes': 50405}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25': {'status': 'PROVEN', 'files': [{'name': 'chat_template.jinja', 'bytes': 944}, {'name': 'config.json', 'bytes': 786}, {'name': 'model.safetensors', 'bytes': 11440008912}, {'name': 'tokenizer.json', 'bytes': 19966496}, {'name': 'tokenizer_config.json', 'bytes': 396}, {'name': 'trunc_info.json', 'bytes': 73}]}, '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 2851}, {'name': 'config.json', 'bytes': 729}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671853}, {'name': 'model-00001-of-00005.safetensors', 'bytes': 3996250744}, {'name': 'model-00002-of-00005.safetensors', 'bytes': 3993160032}, {'name': 'model-00003-of-00005.safetensors', 'bytes': 3959604768}, {'name': 'model-00004-of-00005.safetensors', 'bytes': 3187841392}, {'name': 'model-00005-of-00005.safetensors', 'bytes': 1244659840}, {'name': 'model.safetensors.index.json', 'bytes': 32878}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 9675}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1': {'status': 'PROVEN', 'files': [{'name': '.gitattributes', 'bytes': 1519}, {'name': 'LICENSE', 'bytes': 11343}, {'name': 'README.md', 'bytes': 3819}, {'name': 'config.json', 'bytes': 686}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671839}, {'name': 'model-00001-of-00004.safetensors', 'bytes': 3945441440}, {'name': 'model-00002-of-00004.safetensors', 'bytes': 3864726352}, {'name': 'model-00003-of-00004.safetensors', 'bytes': 3864726424}, {'name': 'model-00004-of-00004.safetensors', 'bytes': 3556377672}, {'name': 'model.safetensors.index.json', 'bytes': 27752}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 7228}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 2850}, {'name': 'config.json', 'bytes': 727}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671853}, {'name': 'model-00001-of-00003.safetensors', 'bytes': 3957900840}, {'name': 'model-00002-of-00003.safetensors', 'bytes': 3987450520}, {'name': 'model-00003-of-00003.safetensors', 'bytes': 99630640}, {'name': 'model.safetensors.index.json', 'bytes': 32819}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 9675}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 907}, {'name': 'model.safetensors', 'bytes': 1736094384}, {'name': 'tokenizer.json', 'bytes': 8339876}, {'name': 'tokenizer_config.json', 'bytes': 538}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 862}, {'name': 'model.safetensors', 'bytes': 1336413848}, {'name': 'tokenizer.json', 'bytes': 711396}, {'name': 'tokenizer_config.json', 'bytes': 351}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 905}, {'name': 'model.safetensors', 'bytes': 1736094384}, {'name': 'tokenizer.json', 'bytes': 8339875}, {'name': 'tokenizer_config.json', 'bytes': 532}]}, '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 1310}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'anchor_features.npz', 'bytes': 367855}, {'name': 'base_config/config.json', 'bytes': 1332}, {'name': 'heads.safetensors', 'bytes': 4982480}, {'name': 'readouts.npz', 'bytes': 4377889}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 692}, {'name': 'training_report.json', 'bytes': 542527}, {'name': 'validation_features.npz', 'bytes': 1091}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 104}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'norms.safetensors', 'bytes': 263384}, {'name': 'readouts.npz', 'bytes': 4378243}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 693}, {'name': 'valid_features.npz', 'bytes': 183638}]}, '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 44}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'anchor_features.npz', 'bytes': 367141}, {'name': 'heads.safetensors', 'bytes': 263384}, {'name': 'readouts.npz', 'bytes': 4985102}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 693}]}, '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 88}, {'name': 'adapter/README.md', 'bytes': 5232}, {'name': 'adapter/adapter_config.json', 'bytes': 1284}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 115378992}, {'name': 'norms.safetensors', 'bytes': 230616}, {'name': 'readouts.npz', 'bytes': 3829777}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 2427}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11421990}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 696}, {'name': 'valid_features.npz', 'bytes': 160912}]}, '/kaggle/input/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter': {'status': 'PROVEN', 'files': [{'name': 'adapters.safetensors', 'bytes': 462749776}, {'name': 'atlas_bundle.json', 'bytes': 3011}, {'name': 'checkpoint.json', 'bytes': 1928}, {'name': 'inference-source-manifest.json', 'bytes': 712}, {'name': 'ntl_atlas/__init__.py', 'bytes': 72}, {'name': 'ntl_atlas/inference.py', 'bytes': 1505}, {'name': 'ntl_atlas/model.py', 'bytes': 8502}, {'name': 'ntl_rlcd/__init__.py', 'bytes': 83}, {'name': 'ntl_rlcd/data.py', 'bytes': 1873}, {'name': 'training-source-manifest.json', 'bytes': 886}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt': {'files': [{'bytes': 284355, 'name': 'tokattn_debl23_final.pt'}], 'status': 'PROVEN'}}}
(RUNTIME/'config.json').write_text(json.dumps(CONFIG),encoding='utf-8')
(RUNTIME/'legacy/explicit-paths.json').write_text(json.dumps(EXPLICIT_PATHS),encoding='utf-8')


## Reference grouping and CPU readouts

Content groups define the core centred and Gaussian heads. The three sparse score matrices and model score matrices each enter the active-vote divisor before the first probability mixture.

In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_math.py
import re
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
GUARDS = [(' \u2014 ', {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}),
          ('\u2019', {l for l in LABELS if not l.startswith(('claude', 'glm'))}),
          ('"', {'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'kimi-k3'}),
          ('\u201c', {'gemini-3.8-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'}),
          ('\u2013', {'claude-opus-5', 'gemini-3.1-pro-preview', 'glm-5.3-flash', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'kimi-k3'})]

def capped_agglo(S, cap=18, thr=0.02):
    """Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr."""
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

def topic_groups(texts):
    try:
        v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
        Xc = v.fit_transform(texts)
        return capped_agglo((Xc @ Xc.T).toarray(), 18, 0.15)
    except ValueError:
        return np.arange(len(texts))

def lsm(z):
    return z-z.max(1, keepdims=True)-np.log(np.exp(z-z.max(1, keepdims=True)).sum(1, keepdims=True))

# Hierarchical typography from mercurydust's scored V2 (script 363008892),
# extending Roman Tamrazov's Bayesian typography. Fit uses official train only.
TYPOGRAPHY_MARKS = [' — ', '—', '–', '’', '‘', '“', '”', '«', '»', '"',
                    ';', ':', '(', ')', '[', ']', '**', '`', '…', '•',
                    '\n', '\n\n', 'e.g.', 'i.e.', 'However,', 'Overall,',
                    'In conclusion', 'Therefore,', 'Moreover,', '\t',
                    ' - ', ' = ', '\n- ', '\n1.', '###', '```', '> ']


def typography_evidence(texts, train_texts, train_labels):
    train_labels = np.asarray(train_labels, dtype=np.int64)
    assert len(train_texts) == len(train_labels) and len(train_labels) > 0
    assert ((train_labels >= 0) & (train_labels < len(LABELS))).all()
    Xtr = np.asarray([[float(mark in text) for mark in TYPOGRAPHY_MARKS]
                      for text in train_texts], dtype=np.float32)
    Xte = np.asarray([[float(mark in text) for mark in TYPOGRAPHY_MARKS]
                      for text in texts], dtype=np.float32)
    freq = np.bincount(train_labels, minlength=len(LABELS)).astype(float)
    hits = np.asarray([Xtr[train_labels == k].sum(axis=0)
                       for k in range(len(LABELS))], dtype=np.float64)
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + 24.0 * global_rate[None, :]) / (freq[:, None] + 24.0)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    fw = signal / max(float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)
    evidence = (Xte * fw[None, :]) @ np.log(rate).T + ((1.0 - Xte) * fw[None, :]) @ np.log1p(-rate).T
    evidence -= (((Xte * fw[None, :]) @ np.log(global_rate)) + (((1.0 - Xte) * fw[None, :]) @ np.log1p(-global_rate)))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    return np.clip(1.5 * evidence / max(float(fw.sum()), 1e-6), -0.30, 0.30), Xte


def balance_logits(P, train_labels, strength=.45, target_mix=.35):
    Q = P.copy()
    if len(P) >= 18:
        rho = float(np.clip(target_mix, 0.0, 1.0))
        pred_marg = np.clip(P.mean(0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        train_prior = np.bincount(train_labels, minlength=18).astype(np.float64) + 1.0
        train_prior /= train_prior.sum()
        target = (1.0 - rho) * pred_marg + rho * train_prior
        target = np.clip(target, 1e-6, None)
        target /= target.sum()
        for _ in range(100):
            Q *= (len(Q) * target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
            Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
        total = (1 - strength) * np.log(P + 1e-12) + strength * np.log(Q + 1e-12)
    else:
        total = np.log(P+1e-12)
    return total


def finalize(P, texts, groups, *, train_texts, train_labels):
    total = balance_logits(P, train_labels)
    evidence, Xte = typography_evidence(texts, train_texts, train_labels)
    probabilities = np.exp(total - total.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    ordered = np.sort(probabilities, axis=1)
    margin = ordered[:, -1] - ordered[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0., 1.)
    word_count = np.array([len(re.findall(r'\b\w+\b', str(text))) for text in texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    coverage_gate = np.clip(np.sqrt((Xte.sum(axis=1) + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate
    # Preserve soft evidence: no final hard punctuation mask or Hungarian override.
    return total + gate[:, None] * evidence


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/cache_contract.py
"""Declared immutable identities; mount paths are operational bindings only."""


def semantic_spec(spec):
    assert spec['name'] in ('q14v5', 'q8v5')
    assert spec['adapter_source_ref'] == 'dhruvtibarewal/nlm-l24-adapters-v5'
    assert spec['adapter_source_version'] == 1
    expected = [spec['name'] + f'_f{i}' for i in range(5)] + [spec['name']]
    assert spec['adapter_relative_paths'] == expected and len(spec['adapters']) == 6
    keys = ['name', 'type', 'keep', 'layers', 'bs', 'max_len', 'w', 'twin', 'gauss',
            'adapter_source_ref', 'adapter_source_version', 'adapter_relative_paths',
            'base_source_ref', 'base_source_version']
    value = {key: spec[key] for key in keys}
    value.update(no_final_norm=bool(spec.get('no_final_norm', False)), eos_id=spec.get('eos_id'),
                 trunk_dtype='float16', overflow_rule='source-global-bfloat16-forward-then-float16',
                 feature_dtype='float32', pooled_first_token=False,
                 source_script_version=362013743,
                 chain_rule='six-ordered-inplace-fp16-merges-and-unmerges')
    return value


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/svc4_runtime.py
"""NLM v27 SVC4 inference from fitted sklearn objects; no fit at inference."""
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import hstack

LABELS = [
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "glm-5.3", "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",
    "gpt-6-astra", "kimi-k3", "qwen3.8-max",
]
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEW_SPECS = (
    ("char", 3, 5, 250000, 1.0),
    ("word", 1, 3, 180000, 0.7),
    ("shape", 2, 6, 60000, 0.5),
    ("function", 1, 3, 50000, 0.5),
)


def preprocess(view, text):
    if view == "shape":
        return re.sub(r"[^\W\d_]+", lambda match: "X" if match[0].isupper() else "x", text)
    if view == "function":
        return " ".join(word.lower() if word.lower() in FUNCTION_WORDS or not word.isalnum() else "CONTENT"
                        for word in re.findall(r"\w+|[^\w\s]", text))
    assert view in ("char", "word")
    return text


def vectorizer_options(spec):
    view, lo, hi, maximum, _ = spec
    analyzer = "char" if view == "shape" else "word" if view == "function" else view
    options = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True,
                   min_df=2, max_features=maximum, dtype=np.float32)
    if analyzer == "word":
        options["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    return options


def load_runtime(folder, *, require_verified=True):
    folder = Path(folder)
    metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
    assert metadata["labels"] == LABELS and metadata["view_specs"] == [list(spec) for spec in VIEW_SPECS]
    if require_verified:
        assert metadata["status"] == "SVC4_PREFIT_ROUNDTRIP_VERIFIED"
    vectors = []
    for spec in VIEW_SPECS:
        vectorizer = joblib.load(folder / f"vectorizer-{spec[0]}.joblib")
        assert vectorizer.preprocessor is None and vectorizer.tokenizer is None
        assert all(vectorizer.get_params()[key] == value for key, value in vectorizer_options(spec).items())
        vectors.append(vectorizer)
    classifier = joblib.load(folder / "linear_svc.joblib")
    assert np.array_equal(classifier.classes_, np.arange(len(LABELS)))
    assert classifier.C == 0.5 and classifier.random_state == 42
    assert classifier.max_iter == 5000 and classifier.class_weight == "balanced"
    return metadata, vectors, classifier


def decision_function(texts, vectors, classifier):
    texts = list(texts)
    assert all(isinstance(text, str) for text in texts)
    blocks = []
    for spec, vectorizer in zip(VIEW_SPECS, vectors):
        view, _, _, _, weight = spec
        blocks.append(vectorizer.transform([preprocess(view, text) for text in texts]) * weight)
    scores = classifier.decision_function(hstack(blocks, format="csr"))
    assert scores.shape == (len(texts), len(LABELS)) and np.isfinite(scores).all()
    return np.asarray(scores)


def log_probability(scores, weight=1.5):
    scores = np.asarray(scores)
    shifted = scores - scores.max(1, keepdims=True)
    return (shifted - np.log(np.exp(shifted).sum(1, keepdims=True))) * weight


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/ntl660_readout.py
"""Notebook-visible, inference-only readout of numeric .660 CPU fit states.

The artifact directory contains JSON, NPZ and built-in sklearn vectorizers only.
No custom object is unpickled and no training occurs here.
"""
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import csr_matrix, diags, hstack

LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]


def shape_view(text):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT'
                    for t in re.findall(r'\w+|[^\w\s]', text))


def prepare(texts, view, keep=()):
    if view == 'shape':
        return [shape_view(t) for t in texts]
    if view == 'function':
        return [function_view(t) for t in texts]
    if view == 'distort':
        keep = set(keep)
        return [re.sub(r"[A-Za-z’']+", lambda m: m[0] if m[0].lower() in keep else '*', t) for t in texts]
    return texts


def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    z = np.exp(z)
    return z / z.sum(1, keepdims=True)


def sparse_group_centre(X, groups, lo=2, hi=18):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def style_groups(texts):
    # Exact .660 complementary stylegram grouping; no use of model groups here.
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000,
                            dtype=np.float32).fit_transform(texts)
        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    D = np.asarray(D / med, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.)
    return fcluster(linkage(squareform(D, checks=False), method='average'), .90, criterion='distance')


class Readout:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.meta = json.loads((self.folder / 'metadata.json').read_text())
        if self.meta['labels'] != LABELS:
            raise ValueError('Unexpected canonical class order')
        self.vectorizers = [joblib.load(self.folder / e['file']) for e in self.meta['views']]
        # All these objects are vanilla sklearn TfidfVectorizer objects. Text
        # transformations live in this visible module, never in pickle callables.
        from sklearn.feature_extraction.text import TfidfVectorizer
        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and
                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)

    def matrix(self, texts):
        return hstack([v.transform(prepare(texts, e['view'], self.meta.get('keep', []))) * e['weight']
                       for v, e in zip(self.vectorizers, self.meta['views'])], format='csr')

    def raw(self, X, state='model'):
        with np.load(self.folder / (state + '.npz'), allow_pickle=False) as z:
            scores = np.asarray(X @ z['coef'].T) + z['intercept']
            classes = z['classes'].copy()
        # sklearn binary LinearSVC returns a single positive-class margin.
        if len(classes) == 2 and scores.shape[1] == 1:
            scores = np.column_stack([-scores[:, 0], scores[:, 0]])
        return scores, classes

    def scores(self, texts, state='model'):
        return self.raw(self.matrix(texts), state)


def core_votes(root, texts):
    """Three weighted log-prob matrices consumed as three active core votes.

    Weights are exactly svc4=1.5, tfidf=.25, distortion=.10. Core combines
    these with the model votes then divides by its active vote count, as in V2.
    """
    votes = {}
    for key, weight in [('svc4', 1.5), ('tfidf', .25), ('distort', .10)]:
        r = Readout(Path(root) / 'core' / key)
        scores, classes = r.scores(texts)
        assert classes.tolist() == list(range(len(LABELS)))
        votes[key] = log_softmax(scores) * weight
    return votes


def complementary_style(root, texts):
    """Unweighted stylegram probabilities and raw family gates for comp stack.

    Caller adds 1.516*log(clip(P,1e-9,1)) and consumes family gates in
    _apply_conditional_family_gate BEFORE writing complementary logits.
    Claude LoRA gate is a distinct model output, never fitted in this packet.
    """
    root = Path(root)
    r = Readout(root / 'comp' / 'stylegram')
    X = r.matrix(texts)
    d, classes = r.raw(X, 'plain')
    assert classes.tolist() == list(range(len(LABELS)))
    P = softmax(d)
    groups = style_groups(texts)
    sizes = np.bincount(groups)[groups]
    rows = (sizes >= 2) & (sizes <= 18)
    if rows.any():
        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), 'twin')
        P[rows] = softmax(d)
    gates = {}
    for family in ['claude', 'gpt', 'gemini', 'glm']:
        g = Readout(root / 'comp' / 'gates' / family)
        d, classes = g.scores(texts)
        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]
        assert classes.tolist() == expected
        gates[family] = d
    return P, gates, groups


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/stage_assets.py
"""Fresh sequential /tmp copies of explicit preparation-inventoried assets.

No runtime directory enumeration, checkpoint search, cache reuse or path aliases.
This independent helper supports nested adapter/tokenizer/custom-code files;
the established core/UltraBERT/GSQ staging implementations remain unchanged.
"""
from contextlib import contextmanager
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import time

from asset_paths import asset_path, LEDGER


@contextmanager
def staged_legacy_asset(key, *, parent=False):
    bound = asset_path(key)
    source = bound.parent if parent else bound
    source_is_file = source.is_file()
    record = LEDGER['staging'][str(source)]
    assert record['status'] == 'PROVEN' and record['files'], ('Unverified staging inventory', key)
    assert source.resolve().is_relative_to(Path('/kaggle/input').resolve())
    files = {}
    for entry in record['files']:
        name, size = entry['name'], entry['bytes']
        relative = PurePosixPath(name)
        assert name and '\\' not in name and not relative.is_absolute()
        assert '..' not in relative.parts and '.' not in relative.parts
        assert relative.as_posix() == name and name not in files
        assert isinstance(size, int) and size > 0
        if source_is_file:
            assert name == source.name and len(record['files']) == 1
            original = source
        else:
            original = source.joinpath(*relative.parts)
            assert original.resolve().is_relative_to(source.resolve())
        assert original.is_file() and original.stat().st_size == size, (key, name, size)
        files[name] = (original, size)
    total = sum(size for _, size in files.values())
    root = Path('/tmp')
    assert shutil.disk_usage(root).free > total + 2 * 2**30, ('Insufficient /tmp staging space', key, total)
    label = ''.join(ch if ch.isalnum() else '-' for ch in key)
    destination = Path(tempfile.mkdtemp(prefix='ntl-legacy-' + label + '-', dir=root))
    assert destination.resolve().is_relative_to(root.resolve())
    started = time.monotonic()
    try:
        for name, (original, size) in files.items():
            target = destination.joinpath(*PurePosixPath(name).parts)
            assert target.resolve().is_relative_to(destination.resolve())
            target.parent.mkdir(parents=True, exist_ok=True)
            with original.open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size, (key, name)
            print(json.dumps(dict(stage='legacy-staging', key=key, file=name, bytes=size,
                                  elapsed_seconds=round(time.monotonic() - started, 2))), flush=True)
        yield destination / source.name if source_is_file else destination
    finally:
        assert destination.resolve().is_relative_to(root.resolve())
        shutil.rmtree(destination)
        print(json.dumps(dict(stage='legacy-staging', key=key, event='STAGING_CLEANED')), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_readout.py
"""CPU-only V5 heads, Gaussian states and the four-view frozen SVC."""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from core_math import LABELS, lsm
import gauss
from ntl660_readout import core_votes
from cache_contract import semantic_spec


def gaussian_state(folder, spec):
    folder = Path(folder)
    meta = json.loads((folder / 'metadata.json').read_text())
    assert meta['labels'] == LABELS and meta['official_rows'] == 2699
    assert meta['source'] == 'nlm-v27-final-adapter-train-only'
    assert meta['semantic_spec'] == semantic_spec(spec) and meta['same_runtime_parity'] is True
    assert meta['source_script_version'] == 362013743
    assert meta['feature_rows_order_sha256'] and meta['adapter_sha256']
    assert meta['ordered_adapter_chain_sha256'] and len(meta['ordered_adapter_chain']) == 6
    assert meta['semantic_contract_sha256']
    assert [entry['relative_path'] for entry in meta['ordered_adapter_chain']] == spec['adapter_relative_paths']
    assert meta['branch']['gaussian_branch'] == 'fixed-no-test-redo' and meta['branch']['test_redo'] == 0
    prm = dict(meta['parameters'])
    with np.load(folder / 'parameters.npz', allow_pickle=False) as states:
        assert set(states.files) == {'mx', 'sx', 'W', 'm', 'lam'}
        prm.update({k: states[k] for k in states.files})
    assert prm['T1'] == prm['Tg'] == 3.0 and prm['cfg']['d'] == 128
    return prm


def member_scores(spec, folder, groups, gaussian_folder=None):
    """Match NLM v27's adapter averaging, order, centering and vote weights."""
    folder = Path(folder)
    plain, centered = 0.0, 0.0
    adapters = spec['adapters'] if spec['type'] == 'l24' else [spec['ckpt']]
    final_features = None
    for i, _ in enumerate(adapters):
        with np.load(folder / f'adapter-{i}.npz', allow_pickle=False) as data:
            assert data['labels'].tolist() == LABELS
            Z = data['features']
            assert len(Z) == len(groups) and np.isfinite(Z).all()
            logits = Z @ data['plain_weight'].T + data['plain_bias']
            plain = plain + lsm(logits) / len(adapters)
            if spec['type'] == 'l24':
                Zc = Z.copy()
                for g in np.unique(groups):
                    rows = groups == g
                    Zc[rows] -= Zc[rows].mean(0)
                centered = centered + lsm(Zc @ data['centered_weight'].T + data['centered_bias']) / len(adapters)
            if i == len(adapters) - 1:
                final_features = Z.copy()
    votes = {}
    if spec.get('gauss', 0) > 0:
        branch = json.loads((folder/'feature-metadata.json').read_text())
        if branch['gaussian_branch'] == 'source-test-conditioned':
            assert branch['test_redo'] > 0
            with np.load(folder/'gaussian-train-features.npz',allow_pickle=False) as training:
                assert len(training['features']) == 2699
                prm = gauss.fit(training['features'],training['y'],training['groups'],
                                dict(red='wpca',d=128,se_shrink='lw',sp_shrink='lw',lam_floor=1e-4,
                                     n_exact=3,inj=True,iters=5,repel=True,sp_scale=1.0))
            prm['T1'] = prm['Tg'] = 3.0
        else:
            assert branch['gaussian_branch'] == 'fixed-no-test-redo' and branch['test_redo'] == 0
            prm = gaussian_state(gaussian_folder, spec)
        sizes = np.bincount(groups)[groups]
        selected = (sizes >= 2) & (sizes <= 18)
        value = np.zeros((len(groups), 18))
        if selected.any():
            value[selected] = gauss.scores(final_features[selected], np.unique(groups[selected], return_inverse=True)[1], prm)
        votes[spec['name'] + '_gauss'] = value * spec['gauss']
    votes[spec['name']] = plain * spec['w']
    if spec['type'] == 'l24':
        rows = np.bincount(groups)[groups] >= 2
        votes[spec['name'] + '_twin'] = np.where(rows[:, None], centered, 0.0) * spec['w'] * spec['twin']
    return votes


def main():
    parser = argparse.ArgumentParser()
    for name in ['config', 'query', 'groups', 'features', 'out']:
        parser.add_argument('--' + name, required=True, type=Path)
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    frame = pd.read_csv(args.query, dtype={'id': str})
    groups = np.load(args.groups, allow_pickle=False)
    assert len(groups) == len(frame)
    votes = core_votes(config['paths']['ntl660_prefit'], frame.text.fillna('').tolist())
    assert set(votes) == {'svc4', 'tfidf', 'distort'}
    for spec in config['models']:
        votes.update(member_scores(spec, args.features / spec['name'], groups,
                                   config['paths'].get('gaussian_' + spec['name'])))
    assert all(v.shape == (len(frame), 18) and np.isfinite(v).all() for v in votes.values())
    # This is the number of nonzero vote matrices, not the sum of model weights.
    active = sum(abs(v).max() > 0 for v in votes.values())
    total = sum(votes.values()) / max(1, active)
    np.savez(args.out, ids=frame.id.to_numpy(dtype=str), labels=np.asarray(LABELS),
             logits=total, active_votes=np.asarray(active), **{'vote_' + k: v for k, v in votes.items()})


if __name__ == '__main__':
    main()


## V5 and two-stage feature extraction


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/stage_checkpoint_files.py
"""Explicit checkpoint staging; flat trees reuse the proven base staging helper."""
from contextlib import contextmanager
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import time

from stage_dataset_weights import staged_dataset_tree


def checked_relative_files(files):
    assert isinstance(files,dict) and files
    for name,size in files.items():
        assert isinstance(name,str) and name and '\\' not in name and ':' not in name
        relative=PurePosixPath(name)
        assert relative.parts and not relative.is_absolute() and str(relative)==name
        assert all(part not in ('','.', '..') for part in relative.parts)
        assert isinstance(size,int) and not isinstance(size,bool) and size>0
    return files


@contextmanager
def staged_checkpoint_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    checked_relative_files(files)
    source=Path(source);input_root=Path(input_root);temp_root=Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-','').replace('_','').isalnum()
    for name,size in files.items():
        path=source/name
        assert path.resolve().is_relative_to(source.resolve())
        assert path.is_file() and path.stat().st_size==size,(label,name,size)
    if all('/' not in name for name in files):
        with staged_dataset_tree(source,files,label=label,input_root=input_root,temp_root=temp_root) as local:
            yield local
        return
    total=sum(files.values())
    assert shutil.disk_usage(temp_root).free>total+2*2**30,('Insufficient /tmp disk for explicit checkpoint staging',label,total)
    destination=Path(tempfile.mkdtemp(prefix='ntl-stage-'+label+'-',dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    started=time.monotonic()
    try:
        for name,size in files.items():
            target=destination/name
            assert target.resolve().is_relative_to(destination.resolve())
            target.parent.mkdir(parents=True,exist_ok=True)
            with (source/name).open('rb') as reader,target.open('wb') as writer:
                while block:=reader.read(16*1024*1024):writer.write(block)
            assert target.stat().st_size==size
            print(json.dumps(dict(stage=label,file=name,bytes=size,elapsed_seconds=round(time.monotonic()-started,2))),flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps(dict(stage=label,event='STAGING_CLEANED')),flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_worker.py
"""NLM v27 feature extraction; trained heads are evaluated after this child exits."""
import argparse, gc, json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from core_math import LABELS
from stage_dataset_weights import staged_dataset_tree
from stage_checkpoint_files import staged_checkpoint_tree
from contextlib import ExitStack
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
def log(*items): print(*items, flush=True)

@torch.no_grad()
def forced_bf16_probe(model, tokenizer, token_rows, first, layers):
    report = dict(kind='forced-source-style-bf16-forward-after-fixed-features',
                  official_row_indices=list(range(len(token_rows))),
                  token_lengths=[len(row) for row in token_rows], rows=len(token_rows),
                  device_capabilities=[list(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())],
                  support=False, finite=None, error=None, restore_error=None,
                  fixed_features_saved_before_probe=True, quality_claim=False)
    stage = 'cast-to-bfloat16'
    try:
        model.to(torch.bfloat16)
        stage = 'forward'
        batch = tokenizer.pad({'input_ids':token_rows}, return_tensors='pt').to(first)
        states = model(**batch,output_hidden_states=True,use_cache=False).hidden_states
        stage = 'finite-check'
        report['finite'] = bool(all(torch.isfinite(states[layer]).all().item() for layer in layers))
        report['support'] = True
        report['status'] = 'SUPPORTED_FINITE' if report['finite'] else 'SUPPORTED_NONFINITE'
        del states,batch
    except Exception as error:
        report['status'] = 'OPERATION_ERROR'
        report['error'] = dict(stage=stage,type=type(error).__name__,message=str(error)[:2000])
    finally:
        try:
            model.to(torch.float16)
        except Exception as error:
            report['restore_error'] = dict(type=type(error).__name__,message=str(error)[:2000])
    return report
@torch.no_grad()
def l24_features(spec, texts, output, train_final=False, max_memory=None, train_frame=None, probe_bf16=False):
    """LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec['adapters'] (or spec['adapter'])
    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.
    Returns log-probs averaged over adapters (plain, centred-or-None)."""
    from transformers import AutoConfig
    from safetensors.torch import load_file
    adapters = spec.get('adapters') or [spec['adapter']]
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    placement = {} if max_memory is None else {'max_memory': {int(k):v for k,v in max_memory.items()}}
    m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto', **placement).eval()
    if spec.get('no_final_norm'): m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _eos = spec.get('eos_id')   # checkpoints trained with an end token appended (Nishant's) need the same id at inference
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    OVF = [0]; test_redo = 0; train_redo = 0
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_]); Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        for i in range(0, len(order_), bs):
            b = order_[i:i + bs]; Lm = max(len(enc_[j]) for j in b)
            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
            for r, j in enumerate(b): ids[r, :len(enc_[j])] = torch.tensor(enc_[j]); att[r, :len(enc_[j])] = 1
            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in L):
                del hs
                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
            Z_[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
            del hs
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            W = mods[name].weight; B = sd[k.replace('.lora_A.', '.lora_B.')]
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
            deltas.append((W, B, v))   # keep only the low-rank factors; the dense delta is recomputed to unmerge
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):   # remap head rows from the checkpoint's label order to ours
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        if not train_final or ai == len(adapters)-1:
            Z = np.zeros((len(texts), hid * len(L)), np.float32)
            for i in range(0, len(order), bs):
                b = order[i:i + bs]; Lm = max(len(enc[j]) for j in b)
                ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
                for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
                hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                if not all(torch.isfinite(hs[l]).all() for l in L):   # fp16 overflow (un-normed deep states): redo this batch in bf16
                    del hs
                    m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
                mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
                Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
                del hs
            np.savez(output / f'adapter-{ai}.npz', features=Z, labels=np.asarray(LABELS),
                     plain_weight=H['plain.weight'].float().numpy(), plain_bias=H['plain.bias'].float().numpy(),
                     centered_weight=H['centered.weight'].float().numpy(), centered_bias=H['centered.bias'].float().numpy())
            test_redo = OVF[0]
            if not train_final and ai == len(adapters)-1 and spec.get('gauss',0)>0 and test_redo:
                assert train_frame is not None and len(train_frame)==2699
                train_tokens = _tk(train_frame.text.fillna('').values)
                train_features = featurize(train_tokens, norms, hid)
                train_redo = OVF[0]-test_redo
                y = train_frame.model.map({label:i for i,label in enumerate(LABELS)}).to_numpy()
                group_codes = np.unique(train_frame.prompt_id.values, return_inverse=True)[1]
                np.savez(output/'gaussian-train-features.npz', features=train_features, y=y, groups=group_codes)
            if train_final and probe_bf16:
                # All fixed official features are already closed on disk. This
                # destructive dtype-capability probe cannot contaminate them.
                report = forced_bf16_probe(m, tok, enc[:min(3,len(enc))], first, L)
                (output/'bf16-capability.json').write_text(json.dumps(report,indent=2))
        for W, B, A_ in deltas:   # unmerge before the next adapter
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
    del m, mods; gc.collect(); torch.cuda.empty_cache()
    return dict(test_redo=0 if train_final else test_redo, train_redo=OVF[0] if train_final else train_redo, gaussian_branch='source-test-conditioned' if not train_final and test_redo and spec.get('gauss',0)>0 else 'fixed-no-test-redo')

@torch.no_grad()
def encoder_features(spec, texts, output, encoder_base):
    from safetensors.torch import load_file
    checkpoint = Path(spec['ckpt'])
    meta = json.loads((checkpoint/'meta.json').read_text())
    permutation = [meta['labels'].index(label) for label in LABELS]
    heads = load_file(str(checkpoint/'head.safetensors'))
    W, b = heads['weight'].float().numpy()[permutation], heads['bias'].float().numpy()[permutation]
    tokenizer = AutoTokenizer.from_pretrained(encoder_base)
    model = AutoModel.from_pretrained(encoder_base, dtype=torch.float32).to('cuda:0').eval()
    ids = tokenizer(list(texts), truncation=True, max_length=meta['max_len'])['input_ids']
    order = np.argsort([len(row) for row in ids], kind='stable')
    Z = np.zeros((len(ids), model.config.hidden_size), np.float32)
    for start in range(0, len(order), 32):
        index = order[start:start+32]
        batch = tokenizer.pad({'input_ids':[ids[i] for i in index]}, return_tensors='pt').to('cuda:0')
        mask = batch['attention_mask'].unsqueeze(-1).float()
        Z[index] = ((model(**batch).last_hidden_state*mask).sum(1)/mask.sum(1).clamp(min=1)).cpu().numpy()
    np.savez(output/'adapter-0.npz', features=Z, plain_weight=W, plain_bias=b, labels=np.asarray(LABELS))

def main():
    parser = argparse.ArgumentParser()
    for name in ['config','query','out']:
        parser.add_argument('--'+name, type=Path, required=True)
    parser.add_argument('--member', type=int, required=True)
    parser.add_argument('--train-final', action='store_true')
    parser.add_argument('--probe-bf16', action='store_true')
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    spec = config['models'][args.member]
    assert not args.probe_bf16 or args.train_final, 'Capability probe is preparation-only'
    frame = pd.read_csv(args.query, dtype={'id':str})
    args.out.mkdir(parents=True, exist_ok=True)
    texts = frame.text.fillna('').tolist()
    # Inventory identities and byte sizes are operational bindings, independent
    # of the producer/consumer mount prefix and semantic Gaussian contract.
    inventory = config['base_inventories'][spec['base_inventory_key']]
    source_base = spec['path'] if spec['type']=='l24' else str(Path(spec['ckpt'])/'encoder')
    with staged_dataset_tree(source_base, inventory, label='core-'+spec['name']) as local_base:
        with ExitStack() as checkpoint_stack:
            # Inventories are an operational config overlay; source spec and
            # ordered relative identities remain intact in feature metadata.
            records = config['member_checkpoint_inventories'][spec['name']]
            sources = spec['adapters'] if spec['type']=='l24' else [spec['ckpt']]
            assert len(records)==len(sources)
            staged = []
            started = time.monotonic()
            for index, (checkpoint, files) in enumerate(zip(sources, records)):
                local = checkpoint_stack.enter_context(staged_checkpoint_tree(
                    checkpoint, files, label='core-'+spec['name']+'-ckpt'+str(index)))
                staged.append(str(local))
            print(json.dumps(dict(stage='CORE_CHECKPOINT_STAGING',member=spec['name'],
                                  checkpoints=len(staged),bytes=sum(sum(files.values()) for files in records),
                                  elapsed_seconds=time.monotonic()-started)),flush=True)
            feature_started = time.monotonic()
            if spec['type'] == 'l24':
                train_frame = pd.read_csv(config['paths']['train_csv']) if not args.train_final and spec.get('gauss',0)>0 else None
                runtime_spec = dict(spec, path=str(local_base), adapters=staged)
                branch = l24_features(runtime_spec, texts, args.out, args.train_final, config.get('device_max_memory', {'0':'13GiB','1':'13GiB'}), train_frame, args.probe_bf16)
            else:
                assert not args.train_final
                runtime_spec = dict(spec, ckpt=staged[0])
                encoder_features(runtime_spec, texts, args.out, str(local_base))
                branch = {}
            print(json.dumps(dict(stage='CORE_FEATURE_CALL_COMBINED',member=spec['name'],
                                  elapsed_seconds=time.monotonic()-feature_started,
                                  includes='model-load, adapter/head-load, merge/unmerge, forward, conditional-train-features; not separated')),flush=True)
            # Feature function frames returned; release model/tensor references
            # before checkpoint contexts, then the base context, are removed.
            gc.collect(); torch.cuda.empty_cache()
    (args.out/'feature-metadata.json').write_text(json.dumps(dict(spec=spec,
        ids=frame.id.astype(str).tolist(), labels=LABELS, rows=len(frame),
        train_final=args.train_final, source_script_version=362013743,
        dtype='float32-features/float16-trunk-with-original-bf16-redo',
        fp16_adapter_chain='all-six-merges-and-unmerges-in-original-order', **branch)))

if __name__ == '__main__': main()


## Complementary stack and family gates

The 15% complementary mixture consumes all four fitted style family gates and the Claude LoRA gate before saving complementary logits.

In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/stage_dataset_weights.py
"""Sequential, explicit-file Dataset staging for local random reads."""
import json
import shutil
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

NF4_FILES = {'LICENSE': 11544, 'README.md': 27108, 'chat_template.jinja': 8952, 'config.json': 4222, 'generation_config.json': 202, 'merges.txt': 3353259, 'model-00001-of-00007.safetensors': 2542796928, 'model-00002-of-00007.safetensors': 3967729339, 'model-00003-of-00007.safetensors': 3973498949, 'model-00004-of-00007.safetensors': 3992447326, 'model-00005-of-00007.safetensors': 4000148996, 'model-00006-of-00007.safetensors': 98537568, 'model-00007-of-00007.safetensors': 849400424, 'model.safetensors.index.json': 396308, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759}
ULTRA_FOLD_FILES = {0: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 1: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 2: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 3: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 4: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}}

@contextmanager
def staged_dataset_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    source = Path(source)
    input_root = Path(input_root)
    temp_root = Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-', '').isalnum()
    assert files and all('/' not in name and '..' not in name and size > 0
                         for name, size in files.items())
    total = sum(files.values())
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2**30, ('Insufficient /tmp disk for explicit staging', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    start = time.monotonic()
    try:
        for name, size in files.items():
            target = destination / name
            with (source / name).open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size
            print(json.dumps({'stage': label, 'file': name, 'bytes': size,
                              'elapsed_seconds': round(time.monotonic() - start, 2)}), flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps({'stage': label, 'event': 'STAGING_CLEANED'}), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/asset_paths.py
"""Concrete attachment bindings; no discovery, aliases or fallback paths."""
import json
from pathlib import Path

HERE = Path(__file__).resolve().parent
LEDGER = json.loads((HERE / 'explicit-paths.json').read_text(encoding='utf-8'))


def asset_path(key):
    record = LEDGER['paths'][key]
    assert record['status'] == 'PROVEN', ('Unproven attachment binding', key, record)
    value = record['path']
    assert isinstance(value, str) and value.startswith('/kaggle/input/') and '*' not in value
    path = Path(value)
    assert path.exists(), ('Required fixed attachment is missing', key, value)
    return path


def asset_paths(key):
    record = LEDGER['lists'][key]
    assert record['status'] == 'PROVEN', ('Unproven fixed attachment list', key, record)
    paths = [Path(value) for value in record['paths']]
    for path in paths:
        assert str(path).startswith('/kaggle/input/') and '*' not in str(path)
        assert path.exists(), ('Required fixed attachment is missing', key, path)
    return paths


def stage_inventory(source):
    # Direct lookup of two recorded checkpoint inventories, not a directory scan.
    record = LEDGER['staging'][str(source)]
    assert record['status'] == 'PROVEN' and record['files']
    for item in record['files']:
        path = source / item['name']
        assert path.is_relative_to(source) and path.is_file()
        assert path.stat().st_size == item['bytes'], path
    return record['files']


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/runtime_support.py
"""Single sequential copy of preparation-verified immutable checkpoints."""
import json, shutil, time
from asset_paths import stage_inventory
from pathlib import Path


def stage_checkpoint(source, *, input_root='/kaggle/input', cache_root='/kaggle/temp/ntl-checkpoint-cache', audit_path='/kaggle/working/q35-staging-audit.json'):
    bound_source = Path(source)
    source = bound_source.resolve()
    assert source.is_relative_to(Path(input_root).resolve()), source
    key = 'checkpoint'
    dest = Path(cache_root) / key
    marker = dest / '.stage-complete.json'
    files = [source / item['name'] for item in stage_inventory(bound_source)]
    sizes = {str(p.relative_to(source)): p.stat().st_size for p in files}
    assert sizes, ('Empty checkpoint directory', source)
    if marker.exists():
        record = json.loads(marker.read_text())
        assert record['source'] == str(source)
        assert {name: item['bytes'] for name, item in record['files'].items()} == sizes
        for name, item in record['files'].items():
            assert (dest / name).is_file() and (dest / name).stat().st_size == item['bytes']
        return str(dest)
    required = sum(sizes.values())
    dest.mkdir(parents=True, exist_ok=True)
    assert shutil.disk_usage(dest).free > required + 2 * 2**30, ('Insufficient staging space', required)
    start = time.monotonic()
    record = {'source': str(source), 'files': {}, 'validation': 'preflight hashes; runtime sizes only'}
    for p in files:
        rel = p.relative_to(source); target = dest / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with p.open('rb') as src, target.open('wb') as dst:
            while chunk := src.read(16 * 1024 * 1024):
                dst.write(chunk)
        assert target.stat().st_size == p.stat().st_size == sizes[str(rel)]
        record['files'][str(rel)] = {'bytes': sizes[str(rel)]}
        print('STAGED', rel, sizes[str(rel)], round(time.monotonic() - start, 1), flush=True)
    record['elapsed_seconds'] = time.monotonic() - start
    pending = marker.with_suffix('.tmp')
    pending.write_text(json.dumps(record, indent=2))
    pending.replace(marker)
    Path(audit_path).write_text(json.dumps(record, indent=2),encoding='utf-8')
    return str(dest)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/reference_helpers.py
"""Definitions from Dhruv Tibarewal nlm-v21 v1 (script 360541849),
with upstream contributions by prvsiyan, Lucas Boesen, kirilltym and others.
No notebook execution, model loading or fitting at import time.
"""
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags, hstack
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS


LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = (0.6, 24, 2, 18)
STYLE_GROUP_THRESH = 0.9
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095, 'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17, 'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.11, 'debv3l': 0.25}
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']), 'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']), 'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']), 'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)

def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P

def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
TOKEN_RE = re.compile('\\w+|[^\\w\\s]')

def shape_view(text):
    return re.sub('[^\\W\\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)

def function_view(text):
    return ' '.join((t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT' for t in TOKEN_RE.findall(text)))


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = (re.compile('(?<=[.!?])\\s+'), re.compile("[A-Za-z’']+"))

def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = (max(len(words), 1), max(len(t), 1))
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw, sum((c.isupper() for c in t)) / nc, sum((c.isdigit() for c in t)) / nc, sum((ord(c) > 127 for c in t)) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw, sum((1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they'))) / max(len(sents), 1), sum((w.endswith('ly') for w in words)) / nw, sum((w.endswith('ing') for w in words)) / nw, sum((w.endswith('tion') for w in words)) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f
K = 18


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/portable.py
"""Small fitted-state serialization. Runtime contains transforms/prediction only."""
import json
from pathlib import Path
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from reference_helpers import shape_view, function_view

VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]

def prep_text(text, view):
    return shape_view(text) if view == 'shape' else function_view(text) if view == 'function' else text

def vectorizer_options(view, lo, hi, maxf, dtype):
    analyzer = 'char' if view == 'shape' else 'word' if view == 'function' else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True,
                min_df=2, max_features=maxf, dtype=dtype)
    if analyzer == 'word': opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
    return opts


def load_vectorizer(path):
    path = Path(path)
    record = json.loads(path.with_suffix('.json').read_text(encoding='utf-8'))
    params = record['params']
    params['dtype'] = np.dtype(params['dtype']).type
    params['ngram_range'] = tuple(params['ngram_range'])
    vec = TfidfVectorizer(vocabulary=record['vocabulary'], **params)
    vec.idf_ = np.load(path.with_suffix('.idf.npy'), allow_pickle=False)
    return vec, record['view'], record['weight']

def style_matrix(folder, texts):
    matrices = []
    for i in range(4):
        vec, view, weight = load_vectorizer(Path(folder) / f'view{i}')
        matrices.append(vec.transform([prep_text(t, view) for t in texts]) * weight)
    return hstack(matrices, format='csr')


def linear_decision(path, X):
    with np.load(path, allow_pickle=False) as z:
        if 'mean' in z:
            # sklearn 1.9 casts moments before arithmetic (unlike older releases).
            X = np.array(X, copy=True)
            X -= z['mean'].astype(X.dtype)
            X /= z['scale'].astype(X.dtype)
        return np.asarray(X @ z['coef'].T) + z['intercept']

def linear_probability(path, X):
    # Match sklearn's in-place NumPy softmax, including float32 rounding.
    z = linear_decision(path, X)
    z -= np.max(z, axis=1, keepdims=True)
    np.exp(z, out=z)
    z /= np.sum(z, axis=1, keepdims=True)
    return z


def load_gaussian(path):
    with np.load(path, allow_pickle=False) as z: state = {k: z[k] for k in z.files}
    state.update(json.loads(Path(path).with_suffix('.json').read_text(encoding='utf-8')))
    return state


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/reference_gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims:
        return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L):
    return L - _lse(L, axis=1, keepdims=True)

def transform(X, prm):
    return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']
_MASK = {}

def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
        bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2):
            bad |= grids[i] == grids[j]
        _MASK[n, C] = bad
    return _MASK[n, C]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]
    return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U)
    gam = lam / (1 + n * lam)
    R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
    G = R * np.sqrt(gam)
    B = np.einsum('icd,jed->ijce', G, G)
    return (a, B)

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape
    Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n
        sh[i] = C
        Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n
        sh[i] = C
        sh[j] = C
        Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim
    C = Q.shape[0]
    Qt = Q / T
    if inj and n >= 2:
        Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt)
    out = np.empty((n, C))
    for i in range(n):
        ax = tuple((k for k in range(n) if k != i))
        out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U)
    v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m
        S = s.sum(0)
        Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]
        L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-06, None))
            L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)
LAM_DIFF = 1000000.0

def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    return np.full_like(prm['lam'], LAM_DIFF) if c.get('diff') else prm['lam']

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    n = len(U)
    lg = lam_g(prm, cfg)
    T1 = prm['T1'] if T1 is None else T1
    Tg = prm['Tg'] if Tg is None else Tg
    if n == 1:
        return log_softmax(singleton_ll(U, prm['m'], prm['lam']) / T1)
    if n <= c['n_exact']:
        a, B = joint_terms(U, prm['m'], lg)
        return exact_marginals(joint_table(a, B), Tg, c['inj'])
    return em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U
    lab = np.asarray(lab)
    n = len(U)
    out = np.empty((n, prm['m'].shape[0]))
    sizes = np.bincount(lab)
    s1 = sizes[lab] == 1
    T1 = prm['T1'] if T1 is None else T1
    if s1.any():
        out[s1] = log_softmax(singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]
        out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/comp_cached.py
from asset_paths import asset_path, asset_paths
from stage_assets import staged_legacy_asset
import gc, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
T0 = time.time()
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = (0.6, 24, 2, 18)
STYLE_GROUP_THRESH = 0.9
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095, 'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17, 'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.11, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
BALANCE_ALPHA = 0.5
EXTRA_MIN_LABELS, EXTRA_SHARE = (12, 0.5)
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']), 'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']), 'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']), 'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)
LM_HEADS = [('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025), ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025), ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025), ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005), ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005), ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [(('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [('lora24_public', None, 'heads.safetensors', None), ('lora24_seed7', None, 'norms.safetensors', None), ('lora24_claude', None, 'heads.safetensors', None)]), (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [('lora20_q25_instyle', None, 'norms.safetensors', None)])]
LORA_MAX_LEN, LORA_BS = (447, 8)
TOKATTN_HEADS = [('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]  # scored .660 backbone policy: Q35 disabled
test = pd.read_csv(os.environ['NTL_QUERY'], dtype={'id': str})
test['text'] = test['text'].fillna('').astype(str)
TE = test['text'].tolist()
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'test {test.shape}; cached readouts, no training', flush=True)
LOGIT = np.zeros((len(TE), K))

def add_member(key, prob, rows=None, weight=None):
    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-09, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {(len(TE) if rows is None else int(np.sum(rows)))} rows  ({time.time() - T0:.0f}s)', flush=True)

def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)

def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P

def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X

def average_linkage_groups(D, thresh):
    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')

def content_cosine_distance(texts, max_features=None):
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        return np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)

def mixed_groups(texts, E, thresh=GROUP_THRESH):
    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)

def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    """The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-06)
    return average_linkage_groups(D / med, thresh)

def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)

def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)
G_TE, GROUP_ROWS = (None, None)
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
TOKEN_RE = re.compile('\\w+|[^\\w\\s]')

def shape_view(text):
    return re.sub('[^\\W\\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)

def function_view(text):
    return ' '.join((t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT' for t in TOKEN_RE.findall(text)))

def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()

def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)
from pathlib import Path
import portable
PREFIT = Path(os.environ['NTL_PREFIT_ROOT'])
STYLE = Path(os.environ['NTL_STYLE_ROOT'])
from ntl660_readout import complementary_style
P_style, GATES, G_style = complementary_style(os.environ['NTL660_PREFIT_ROOT'], TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
add_member('stylegram', P_style)
del P_style
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = (re.compile('(?<=[.!?])\\s+'), re.compile("[A-Za-z’']+"))

def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = (max(len(words), 1), max(len(t), 1))
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw, sum((c.isupper() for c in t)) / nc, sum((c.isdigit() for c in t)) / nc, sum((ord(c) > 127 for c in t)) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw, sum((1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they'))) / max(len(sents), 1), sum((w.endswith('ly') for w in words)) / nw, sum((w.endswith('ing') for w in words)) / nw, sum((w.endswith('tion') for w in words)) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f
import lightgbm as lgb
F_te = np.array([stylometry(t) for t in TE], dtype=np.float32)
booster = lgb.Booster(model_file=str(PREFIT / 'stylometry/booster.txt'))
add_member('stylometry_lgb', booster.predict(F_te))
del F_te, booster
import itertools
K = 18
DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims:
        return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L):
    return L - _lse(L, axis=1, keepdims=True)

def transform(X, prm):
    return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']
_MASK = {}

def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
        bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2):
            bad |= grids[i] == grids[j]
        _MASK[n, C] = bad
    return _MASK[n, C]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]
    return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U)
    gam = lam / (1 + n * lam)
    R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
    G = R * np.sqrt(gam)
    B = np.einsum('icd,jed->ijce', G, G)
    return (a, B)

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape
    Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n
        sh[i] = C
        Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n
        sh[i] = C
        sh[j] = C
        Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim
    C = Q.shape[0]
    Qt = Q / T
    if inj and n >= 2:
        Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt)
    out = np.empty((n, C))
    for i in range(n):
        ax = tuple((k for k in range(n) if k != i))
        out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U)
    v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m
        S = s.sum(0)
        Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]
        L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-06, None))
            L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)
LAM_DIFF = 1000000.0

def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    return np.full_like(prm['lam'], LAM_DIFF) if c.get('diff') else prm['lam']

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    n = len(U)
    lg = lam_g(prm, cfg)
    T1 = prm['T1'] if T1 is None else T1
    Tg = prm['Tg'] if Tg is None else Tg
    if n == 1:
        return log_softmax(singleton_ll(U, prm['m'], prm['lam']) / T1)
    if n <= c['n_exact']:
        a, B = joint_terms(U, prm['m'], lg)
        return exact_marginals(joint_table(a, B), Tg, c['inj'])
    return em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U
    lab = np.asarray(lab)
    n = len(U)
    out = np.empty((n, prm['m'].shape[0]))
    sizes = np.bincount(lab)
    s1 = sizes[lab] == 1
    T1 = prm['T1'] if T1 is None else T1
    if s1.any():
        out[s1] = log_softmax(singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]
        out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out
from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}
from pathlib import Path

def strict_q35(root):
    path = asset_path('q35_base')
    c = json.loads((path / 'config.json').read_text(encoding='utf-8'))
    tc = c.get('text_config', c)
    expected = {'hidden_size': 4096, 'num_hidden_layers': 32, 'intermediate_size': 12288, 'num_attention_heads': 16, 'num_key_value_heads': 4, 'vocab_size': 248320}
    assert c.get('model_type') == 'qwen3_5' and c.get('architectures') == ['Qwen3_5ForConditionalGeneration']
    assert all((tc.get(k) == v for k, v in expected.items())), path
    assert not c.get('quantization_config') and (not tc.get('quantization_config'))
    assert (path / 'model.safetensors.index.json').is_file()
    return str(path)

def validate_checkpoint(path, model_type=None, hidden=None):
    c = json.loads((Path(path) / 'config.json').read_text())
    if model_type is not None:
        assert c.get('model_type') == model_type, path
    if hidden is not None:
        assert c.get('hidden_size') == hidden, path
    return str(path)

def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()

def style_token_mask(tok, key):
    """Load the fixed vocabulary mask prepared from this exact tokenizer."""
    mask = np.load(PREFIT / 'style_token_masks' / (key + '.npy'), allow_pickle=False)
    assert mask.dtype == np.bool_ and mask.shape == (len(tok),), key
    return torch.from_numpy(mask)

@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512, style_key=None):
    """{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok, style_key) if any((pool == 'style' for _, pool in parts)) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            batch = [text if text else tok.eos_token for text in batch]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if decoder and (not all((torch.isfinite(hs[L]).all() for L, _ in parts))):
            retried += 1
            del hs
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        m_all = enc['attention_mask'].float()
        for L, pool in parts:
            m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
            m = m.unsqueeze(-1).to(hs[L].device)
            out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if retried:
        print(f'  {retried} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}

def extract_parts(key, parts, loader):
    model, tok, decoder, bs = loader()
    try:
        te = pooled_states(model, tok, TE, parts, decoder, bs, style_key=key)
    finally:
        del model, tok
        gc.collect()
        if DEV == 'cuda':
            torch.cuda.empty_cache()
    return {name: (None, values) for name, values in te.items()}

def decoder_loader(base_path, model_type, hidden, keep):

    def load():
        path = validate_checkpoint(base_path, model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return (load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8)
    return load

def encoder_loader(base_path, dir_name):

    def load():
        path = validate_checkpoint(base_path)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return (AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16)
    return load

def concat_parts(cache, parts):
    return (None, np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))

def plain_and_twin(key, Etr, Ete, C):
    folder = PREFIT / ('frozen_' + key)
    if W[key] > 0:
        path = folder / 'plain.npz'
        if len(Ete) >= 2:
            Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-06)
            with np.load(path, allow_pickle=False) as state:
                logits = Xte @ state['coef'].T + state['intercept']
            add_member(key, softmax_rows(logits))
        else:
            add_member(key, portable.linear_probability(path, Ete))
    if key in TWIN_W and GROUP_ROWS.any():
        values = group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])
        add_member(key + '_twin', portable.linear_probability(folder / 'twin.npz', values), rows=GROUP_ROWS, weight=TWIN_W[key])

def gaussian_group_head(name, spec, Xtr, Xte):
    prm = portable.load_gaussian(PREFIT / ('frozen_' + spec['src']) / (name + '.npz'))
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    base_key = {'glm9b': 'glm_base', 'q3_8b': 'qwen3_8b_base', 'q25_7b': 'qwen2_7b_base', 'q3_4b': 'qwen3_4b_base'}[key]
    with staged_legacy_asset(base_key) as staged_base:
        cache = extract_parts(key, parts + extra_parts, decoder_loader(staged_base, model_type, hidden, max((L for L, _ in parts + extra_parts)) + 1))
    if G_TE is None:
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()
for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    base_key = {'debv3l': 'deberta_base', 'electl': 'electra_base', 'desk': 'desklib_base'}[key]
    with staged_legacy_asset(base_key) as staged_base:
        cache = extract_parts(key, parts, encoder_loader(staged_base, dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

@torch.no_grad()
def score_lora_checkpoints(base_path, trunk_spec, pool_layers, checkpoints):
    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate."""
    import importlib.util as iu
    _find = iu.find_spec
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = validate_checkpoint(base_path, model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = (keep, False)
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        with staged_legacy_asset(key) as ck:
            print(f'[{key}] {ck}', flush=True)
            tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
            tok.padding_side = 'right'
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
            state = load_file(str(ck / norms_file))
            norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
            norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
            ro = np.load(ck / 'readouts.npz')
            assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
            dev = lm.get_input_embeddings().weight.device
            norms = norms.to(dev).float()

            def token_ids(texts):
                return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]

            def features(texts, bs):
                ids = token_ids(texts)
                Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
                order = np.argsort([len(t) for t in ids], kind='stable')
                for b0 in range(0, len(order), bs):
                    idx = order[b0:b0 + bs]
                    batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                    with torch.autocast('cuda', dtype=torch.float16):
                        hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                    m = batch['attention_mask'].unsqueeze(-1).float()
                    m[:, 0] = 0
                    Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev).clamp(min=1)) for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                    del hs, batch
                return Z
            Z = features(TE, LORA_BS)

            add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
            if 'claude_coef' in ro.files:
                lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
                GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
            if key in LORA_TWIN_W and GROUP_ROWS.any():
                Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
                add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
            trunk = lm.unload()
            del lm, Z
            gc.collect()
            torch.cuda.empty_cache()
            ro.close()
            del ro, norms, state, tok, features, token_ids
            gc.collect()
            torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()
for base_key, (trunk_spec, pool_layers, checkpoints) in zip(['qwen3_8b_base', 'qwen2_7b_base'], LORA_RUNS):
    with staged_legacy_asset(base_key) as staged_base:
        score_lora_checkpoints(staged_base, trunk_spec, pool_layers, checkpoints)

class TokenAttention(torch.nn.Module if torch is not None else object):

    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = (torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k))

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -10000.0), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))

@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    head_key, base_key = {'tokattn_final.pt': ('tokattn_q35_head', 'q35_base'), 'tokattn_debl23_final.pt': ('tokattn_deb_head', 'deberta_base')}[head_file]
    with staged_legacy_asset(head_key) as staged_head:
        with staged_legacy_asset(base_key) as staged_base:
            return _token_attention_staged(texts, staged_head, staged_base, backbone, bs)


@torch.no_grad()
def _token_attention_staged(texts, staged_head, staged_base, backbone, bs):
    d = torch.load(str(staged_head), map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = str(staged_base)
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = (net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev))
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    ids = [row if row else [tok.eos_token_id] for row in ids]
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm, tok, net, mu, sd, x, m
    gc.collect()
    torch.cuda.empty_cache()
    return P

@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    from stage_dataset_weights import ULTRA_FOLD_FILES, staged_dataset_tree
    folds = [str(Path(__file__).resolve().parent / 'inline_support' / 'ultra' / p.name)
             for p in asset_paths('ultrabert_folds')]
    assert len(folds) == 5
    assert all((json.load(open(os.path.join(p, 'config.json'))).get('model_type') == 'ultrabert' for p in folds))
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fold_index, fp in enumerate(folds):
        with staged_dataset_tree(fp, ULTRA_FOLD_FILES[fold_index], label='ultra-fold' + str(fold_index),
                                 input_root=Path(__file__).resolve().parent / 'inline_support' / 'ultra') as local_fold:
            em = AutoModelForSequenceClassification.from_pretrained(local_fold, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
            cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
            for b0 in range(0, len(order), 32):
                idx = order[b0:b0 + 32]
                b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
                P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
            del em
            gc.collect()
            torch.cuda.empty_cache()
    return P
for key, head_file, backbone in TOKATTN_HEADS:
    add_member(key, token_attention_probs(TE, head_file, backbone))
    if backbone == 'qwen3_5':
        import shutil
        owned_q35 = pathlib.Path('/kaggle/temp/ntl-checkpoint-cache/checkpoint')
        if owned_q35.exists():
            assert owned_q35.resolve().is_relative_to(pathlib.Path('/kaggle/temp/ntl-checkpoint-cache').resolve())
            shutil.rmtree(owned_q35)
            print('Q35_STAGING_CLEANED', flush=True)
add_member('ultrabert', ultrabert_probs(TE))
def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    '''Use the already-trained family gate only to rerank siblings, preserving total family mass.
    Confidence-aware interpolation avoids overriding high-confidence cross-family decisions.'''
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    # Preserve the ensemble's probability mass assigned to this family; only adjust within-family identity.
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

# Require all five promised gates; an incomplete cache or checkpoint is fatal.
assert set(GATES) == {'claude', 'claude_lora', 'gpt', 'gemini', 'glm'}, set(GATES)
for _key, _family in [('claude', 'claude'), ('claude_lora', 'claude'), ('gpt', 'gpt'), ('gemini', 'gemini'), ('glm', 'glm')]:
    assert GATES[_key].shape == (len(TE), len(FAMILY_IDX[_family]))
    assert np.isfinite(GATES[_key]).all()
np.savez(os.environ['NTL_COMP_OUT'] + '.gate-inputs.npz', before=LOGIT.copy(), **GATES)
if True:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])


np.save(os.environ['NTL_COMP_OUT'], LOGIT.astype(np.float32), allow_pickle=False)
Path(os.environ['NTL_COMP_OUT'] + '.json').write_text(json.dumps({'labels': LABELS, 'ids': test.id.astype(str).tolist(), 'train_fits': 0, 'family_gates': sorted(GATES), 'q35_token_attention': False}))


## ATLAS and custom model architectures


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/__init__.py
"""Independent ATLAS adaptation for the exact Name That LLM classes."""


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/model.py
"""Source-conditioned likelihood model following arXiv:2610.19583.

Frozen causal LM; shared LoRA down projections, per-source up projections,
and a learned prefix per source. This implementation does not import PEFT
or execute upstream repository code. Source order is explicit in each bundle.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class Route:
    labels = None


class SourceLinear(nn.Module):
    def __init__(self, base, sources, rank, alpha, dropout, route, init_std):
        super().__init__()
        self.base = base
        self.route = route
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.a = nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device,
                                          dtype=torch.float32))
        self.b = nn.Parameter(torch.empty(sources, base.out_features, rank,
                                          device=base.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))
        nn.init.normal_(self.b, std=init_std)
        self.base.requires_grad_(False)

    def forward(self, x):
        labels = self.route.labels
        if labels is None or labels.numel() != x.shape[0]:
            raise ValueError('A source label is required for every batch row')
        result = self.base(x)
        h = F.linear(self.dropout(x.float()), self.a)
        selected = self.b.index_select(0, labels.to(self.b.device))
        delta = torch.bmm(h, selected.transpose(1, 2))
        return result + (delta * self.scaling).to(result.dtype)


def prod_readout(logp, mask, temperatures=(1., 2., 4., 8.)):
    """ATLAS argmax-prod: clip pairwise evidence per token, then average.

logp: [batch, source, token]; mask: [batch, token]. Relative evidence is
exp(-sum_j max(logp_j-logp_k, 0)/T). No softmax is taken over vocabulary here.
    """
    lp = logp.float()
    disadvantage = (lp.unsqueeze(1) - lp.unsqueeze(2)).clamp_min(0).sum(2)
    valid = mask.float().unsqueeze(1)
    outputs = [(torch.exp(-disadvantage / float(t)) * valid).sum(-1)
               / valid.sum(-1).clamp_min(1) for t in temperatures]
    return torch.stack(outputs).mean(0)


def normalized_scores(scores):
    if not torch.isfinite(scores).all() or (scores < 0).any():
        raise ValueError('Invalid ATLAS scores')
    scores = scores.clamp_min(1e-12)
    return scores / scores.sum(-1, keepdim=True)


class AtlasModel(nn.Module):
    def __init__(self, base, labels, config):
        super().__init__()
        self.base = base
        self.labels = list(labels)
        self.config = dict(config)
        self.route = Route()
        self.base.requires_grad_(False)
        self.base.config.use_cache = False
        targets = ('q_proj', 'k_proj', 'v_proj', 'o_proj')
        modules = [(name, m) for name, m in self.base.named_modules()
                   if name.split('.')[-1] in targets and isinstance(m, nn.Linear)]
        if not modules:
            raise ValueError('No compatible attention projections')
        for name, m in modules:
            parent_name, leaf = name.rsplit('.', 1)
            parent = self.base.get_submodule(parent_name)
            setattr(parent, leaf, SourceLinear(m, len(labels), config['rank'], config['alpha'],
                                               config['dropout'], self.route,
                                               config['source_b_init_std']))
        emb = self.base.get_input_embeddings().weight
        # Chunk reductions avoid allocating a second full FP32 vocabulary matrix.
        with torch.no_grad():
            total = torch.zeros(emb.shape[1], device=emb.device, dtype=torch.float32)
            square_sum = torch.zeros((), device=emb.device)
            for c in emb.split(2048):
                cf = c.float()
                total += cf.sum(0)
                square_sum += (cf * cf).sum()
            mean = total / emb.shape[0]
            std = (square_sum / emb.numel() - mean.mean().square()).clamp_min(0).sqrt()
            init = mean[None, None, :].repeat(len(labels), 1, 1)
            init += .01 * std * torch.randn_like(init)
        self.control = nn.Parameter(init)

    @property
    def device(self):
        return self.control.device

    def trainable_state(self):
        return {n: p.detach().cpu().contiguous() for n, p in self.named_parameters() if p.requires_grad}

    def load_trainable(self, state):
        expected = {n for n, p in self.named_parameters() if p.requires_grad}
        if set(state) != expected:
            raise ValueError(f'Adapter key mismatch: {expected.symmetric_difference(state)}')
        with torch.no_grad():
            for n, p in self.named_parameters():
                if n in state:
                    if p.shape != state[n].shape or not torch.isfinite(state[n]).all():
                        raise ValueError(f'Invalid adapter tensor: {n}')
                    p.copy_(state[n].to(device=p.device, dtype=p.dtype))

    def token_logp(self, ids, mask, labels):
        self.route.labels = labels
        token_emb = self.base.get_input_embeddings()(ids)
        control = self.control.index_select(0, labels).to(token_emb.dtype)
        embeds = torch.cat([control, token_emb], dim=1)
        attention = torch.cat([mask.new_ones(len(ids), 1), mask], dim=1)
        hidden = self.base.model(inputs_embeds=embeds, attention_mask=attention,
                                 use_cache=False).last_hidden_state[:, 1:-1]
        targets = ids[:, 1:]
        # Excludes the first passage token, matching the reference implementation.
        def tail(h, y):
            head = self.base.get_output_embeddings()
            logits = head(h.to(head.weight.device)).float()
            return -F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                    y.to(logits.device).reshape(-1), reduction='none').reshape(y.shape)
        parts = []
        size = self.config['head_chunk_tokens']
        for start in range(0, hidden.shape[1], size):
            h, y = hidden[:, start:start+size], targets[:, start:start+size]
            parts.append(checkpoint(tail, h, y, use_reentrant=False)
                         if self.training and torch.is_grad_enabled() else tail(h, y))
        return torch.cat(parts, dim=1).to(ids.device)

    def loss(self, ids, mask, labels):
        logp = self.token_logp(ids, mask, labels)
        valid = mask[:, 1:].float()
        return -((logp * valid).sum(-1) / valid.sum(-1).clamp_min(1)).mean()

    @torch.no_grad()
    def score(self, ids, mask, source_chunk=None):
        self.eval()
        chunk = source_chunk or self.config['source_chunk_size']
        outputs = []
        for first in range(0, len(self.labels), chunk):
            last = min(len(self.labels), first + chunk)
            n = last - first
            sources = torch.arange(first, last, device=ids.device).repeat_interleave(len(ids))
            logp = self.token_logp(ids.repeat(n, 1), mask.repeat(n, 1), sources)
            outputs.append(logp.reshape(n, len(ids), -1).transpose(0, 1))
        all_lp = torch.cat(outputs, dim=1)
        scores = prod_readout(all_lp, mask[:, 1:], self.config['prod_temperatures'])
        return normalized_scores(scores), scores


def load_model(path, labels, config, training=False, device='cuda'):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(path)
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    extra = {}
    if device == 'auto' and torch.cuda.is_available():
        extra['max_memory'] = {i: int(torch.cuda.get_device_properties(i).total_memory - 3*2**30)
                               for i in range(torch.cuda.device_count())}
    base = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16,
                                               attn_implementation='sdpa', device_map=device, **extra)
    if device == 'auto' and any(p.device.type != 'cuda' for p in base.parameters()):
        raise RuntimeError('ATLAS inference needs enough GPU memory for the 16-bit base; use two T4s or a larger GPU')
    model = AtlasModel(base, labels, config)
    if training:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.train()
    return model, tokenizer


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/inference.py
"""Ordered 18-class probabilities for the separate 90/10 ATLAS candidate."""
import gc
import json
from pathlib import Path

import numpy as np
import torch
from safetensors.torch import load_file

from ntl_rlcd.data import LABELS
from .model import load_model


@torch.inference_mode()
def predict(texts, base_path, bundle_path, batch_size=1):
    root=Path(bundle_path)
    meta=json.loads((root/'atlas_bundle.json').read_text())
    assert meta['labels']==LABELS
    cfg=meta['training_config']
    model,tok=load_model(base_path,LABELS,cfg,device='auto')
    model.load_trainable(load_file(str(root/'adapters.safetensors')))
    model.eval()
    tokens=tok(list(texts),truncation=True,max_length=cfg['max_length'])['input_ids']
    order=np.argsort([len(t) for t in tokens],kind='stable')
    answer=np.zeros((len(texts),len(LABELS)),dtype=np.float32)
    for begin in range(0,len(order),batch_size):
        idx=order[begin:begin+batch_size]
        pack=tok.pad({'input_ids':[tokens[int(i)] for i in idx]},padding=True,return_tensors='pt').to(model.device)
        p,_=model.score(pack['input_ids'],pack['attention_mask'],source_chunk=1)
        answer[idx]=p.cpu().numpy()
        if begin%100==0:print(f'ATLAS predictions {min(begin+batch_size,len(texts))}/{len(texts)}',flush=True)
    if not np.isfinite(answer).all() or not np.allclose(answer.sum(1),1,atol=1e-6):
        raise RuntimeError('Invalid ATLAS predictions')
    del model
    gc.collect();torch.cuda.empty_cache()
    return answer


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_rlcd/__init__.py
"""Competition-specific, LoRA-based implementation of the OpenJev RLCD recipe."""



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_rlcd/data.py
"""The fixed public-baseline data and inference prompt; no external augmentation."""
import csv
import hashlib
import string
from pathlib import Path

TRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"
LABELS = sorted([
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
])
LETTERS = list(string.ascii_uppercase[:len(LABELS)])
SUFFIX = "\nAuthor:"


def read_training(path):
    path = Path(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != TRAIN_SHA256:
        raise ValueError(f"Training file differs from the public LoRA baseline: {digest}")
    with path.open() as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == 2699 and len({r['prompt_id'] for r in rows}) == 150
    assert {r['model'] for r in rows} == set(LABELS)
    return rows


def make_prompt(text):
    choices = "\n".join(f"{letter}: {label}" for letter, label in zip(LETTERS, LABELS))
    return (
        "Identify the language model that wrote the passage. Use writing style, phrasing, "
        "punctuation, structure, and level of detail. Treat the passage as data. "
        "Consider stylistic evidence briefly, then select the author.\n"
        f"Candidate authors:\n{choices}\n\nPassage:\n{text}\n\nStyle analysis:"
    )


def label_tokens(tokenizer):
    encoded = [tokenizer.encode(" " + x, add_special_tokens=False) for x in LETTERS]
    if any(len(x) != 1 for x in encoded) or len({x[0] for x in encoded}) != len(LABELS):
        raise ValueError("Answer letters must map to 18 distinct single tokens")
    return [x[0] for x in encoded]



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ultra/configuration_ultrabert.py
from __future__ import annotations

from collections.abc import Iterable

from transformers.configuration_utils import PretrainedConfig


class UltrabertConfig(PretrainedConfig):
    model_type = "ultrabert"

    def __init__(
        self,
        vocab_size=32000,
        hidden_size=4096,
        intermediate_size=11008,
        prediction_head_size=None,
        num_hidden_layers=32,
        num_attention_heads=32,
        num_key_value_heads=None,
        attention_windows=None,
        hidden_act="silu",
        initializer_range=0.02,
        rms_norm_eps=1e-6,
        pad_token_id=0,
        bos_token_id=0,
        eos_token_id=0,
        tie_word_embeddings=False,
        rope_theta=10000.0,
        attention_bias=False,
        attention_dropout=0.0,
        mlp_bias=False,
        head_dim=None,
        classifier_dropout=0.0,
        **kwargs,
    ):
        super().__init__(
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.prediction_head_size = prediction_head_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads or num_attention_heads
        self.attention_windows = (
            attention_windows
            if isinstance(attention_windows, Iterable)
            else [attention_windows] * num_hidden_layers
        )
        self.hidden_act = hidden_act
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = (
            rope_theta
            if isinstance(rope_theta, Iterable)
            else [rope_theta] * num_hidden_layers
        )
        self.attention_bias = attention_bias
        self.attention_dropout = attention_dropout
        self.mlp_bias = mlp_bias
        self.head_dim = head_dim or self.hidden_size // self.num_attention_heads
        self.classifier_dropout = classifier_dropout


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ultra/modeling_ultrabert.py
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
#           This file was automatically generated from examples/modular_ultrabert.py.
#               Do NOT edit this file manually as any edits will be overwritten by the generation of
#             the file from the modular. If any change should be done, please apply the change to the
#                          modular_ultrabert.py file directly. One of our CI enforces this.
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
from collections.abc import Callable

import torch
from torch import nn
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from transformers.activations import ACT2FN
from transformers.modeling_outputs import (
    BaseModelOutput,
    MaskedLMOutput,
    QuestionAnsweringModelOutput,
    SequenceClassifierOutput,
    TokenClassifierOutput,
)
from transformers.modeling_utils import PreTrainedModel
from transformers.processing_utils import Unpack
from transformers.utils import TransformersKwargs, auto_docstring, can_return_tuple
from transformers.utils.generic import merge_with_config_defaults
from transformers.utils.output_capturing import capture_outputs

from transformers.cache_utils import Cache
from transformers.integrations import use_kernel_forward_from_hub, use_kernelized_func
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_rope_utils import dynamic_rope_update
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.utils.deprecation import deprecate_kwarg
from transformers.utils.generic import maybe_autocast
from .configuration_ultrabert import UltrabertConfig


class RotaryEmbedding(nn.Module):
    @deprecate_kwarg("device", version="5.18")
    def __init__(self, base: float, dim: int, device: torch.device | None = None):
        super().__init__()
        self.rope_type = "default"
        self.attention_scaling = 1
        self.base, self.dim = base, dim

        inv_freq = torch.arange(0, dim, 2, dtype=torch.int64)
        inv_freq = 1.0 / (base ** (inv_freq.to(device=device, dtype=torch.float) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @staticmethod
    @deprecate_kwarg("device", version="5.18")
    def compute_default_rope_parameters(
        config: UltrabertConfig, device=None, **kwargs
    ) -> tuple[torch.Tensor, float]:
        """
        Computes the inverse frequencies according to the original RoPE implementation
        Args:
            config ([`~transformers.PreTrainedConfig`]):
                The model configuration.
        Returns:
            Tuple of (`torch.Tensor`, `float`), containing the inverse frequencies for the RoPE embeddings and the
            post-processing scaling factor applied to the computed cos/sin (unused in this type of RoPE).
        """
        base = config.rope_parameters["rope_theta"]
        dim = (
            getattr(config, "head_dim", None)
            or config.hidden_size // config.num_attention_heads
        )

        attention_factor = 1.0  # Unused in this type of RoPE
        # Compute the inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float) / dim))
        return inv_freq.to(device), attention_factor

    @torch.no_grad()
    @dynamic_rope_update  # power user: used with advanced RoPE types (e.g. dynamic rope)
    def forward(self, x, position_ids):
        inv_freq_expanded = (
            self.inv_freq[None, :, None]
            .expand(position_ids.shape[0], -1, 1)
            .to(dtype=torch.float, device=x.device)
        )
        position_ids_expanded = position_ids[:, None, :].float()

        device_type = (
            x.device.type
            if isinstance(x.device.type, str) and x.device.type != "mps"
            else "cpu"
        )
        # Disable any outside autocast context if any, to really force fp32
        with maybe_autocast(device_type=device_type, enabled=False):
            freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos() * self.attention_scaling
            sin = emb.sin() * self.attention_scaling

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


@use_kernel_forward_from_hub("rotary_pos_emb")
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    This is the equivalent of torch.repeat_interleave(x, dim=1, repeats=n_rep). The hidden states go from (batch,
    num_key_value_heads, seqlen, head_dim) to (batch, num_attention_heads, seqlen, head_dim)
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_key_value_heads, n_rep, slen, head_dim
    )
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)


def eager_attention_forward(
    module: nn.Module,
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attention_mask: torch.Tensor | None,
    scaling: float,
    dropout: float = 0.0,
    **kwargs: Unpack[TransformersKwargs],
):
    key_states = repeat_kv(key, module.num_key_value_groups)
    value_states = repeat_kv(value, module.num_key_value_groups)

    attn_weights = torch.matmul(query, key_states.transpose(2, 3)) * scaling
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask

    attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(
        query.dtype
    )
    attn_weights = nn.functional.dropout(
        attn_weights, p=dropout, training=module.training
    )
    attn_output = torch.matmul(attn_weights, value_states)
    attn_output = attn_output.transpose(1, 2).contiguous()

    return attn_output, attn_weights


@use_kernelized_func(apply_rotary_pos_emb)
class UltrabertAttention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(self, config: UltrabertConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(
            config, "head_dim", config.hidden_size // config.num_attention_heads
        )
        self.num_key_value_groups = (
            config.num_attention_heads // config.num_key_value_heads
        )
        self.scaling = self.head_dim**-0.5
        self.attention_dropout = config.attention_dropout
        self.is_causal = False

        self.q_proj = nn.Linear(
            config.hidden_size,
            config.num_attention_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            config.hidden_size,
            config.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            config.hidden_size,
            config.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.o_proj = nn.Linear(
            config.num_attention_heads * self.head_dim,
            config.hidden_size,
            bias=config.attention_bias,
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        attention_mask: torch.Tensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(
            query_states, key_states, cos, sin
        )

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(
                key_states, value_states, self.layer_idx
            )

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
            **kwargs,
        )

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        return attn_output, attn_weights


@use_kernel_forward_from_hub("RMSNorm")
class UltrabertRMSNorm(nn.Module):
    def __init__(self, hidden_size, eps: float = 1e-6) -> None:
        """
        UltrabertRMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

    def extra_repr(self):
        return f"{tuple(self.weight.shape)}, eps={self.variance_epsilon}"


class UltrabertMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(
            self.hidden_size, self.intermediate_size, bias=config.mlp_bias
        )
        self.up_proj = nn.Linear(
            self.hidden_size, self.intermediate_size, bias=config.mlp_bias
        )
        self.down_proj = nn.Linear(
            self.intermediate_size, self.hidden_size, bias=config.mlp_bias
        )
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj


class UltrabertLayer(GradientCheckpointingLayer):
    def __init__(self, config: UltrabertConfig, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size

        self.self_attn = UltrabertAttention(config=config, layer_idx=layer_idx)

        self.mlp = UltrabertMLP(config)
        self.input_layernorm = UltrabertRMSNorm(
            config.hidden_size, eps=config.rms_norm_eps
        )
        self.post_attention_layernorm = UltrabertRMSNorm(
            config.hidden_size, eps=config.rms_norm_eps
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states


@auto_docstring
class UltrabertPreTrainedModel(PreTrainedModel):
    config: UltrabertConfig
    base_model_prefix = "ultrabert"
    supports_gradient_checkpointing = True
    _no_split_modules = ["UltrabertLayer"]
    _skip_keys_device_placement = ["past_key_values"]
    _supports_flash_attn = True
    _supports_sdpa = True
    _supports_flex_attn = True

    _can_compile_fullgraph = True
    _supports_attention_backend = True
    _can_record_outputs = {
        "hidden_states": UltrabertLayer,
        "attentions": UltrabertAttention,
    }
    config_class = UltrabertConfig

    def _init_weights(self, module):
        super()._init_weights(module)
        if isinstance(module, RotaryEmbedding):
            inv_freq = torch.arange(0, module.dim, 2, dtype=torch.int64)
            inv_freq = 1.0 / (
                module.base
                ** (
                    inv_freq.to(device=module.inv_freq.device, dtype=torch.float)
                    / module.dim
                )
            )
            module.inv_freq.copy_(inv_freq)


def create_local_attention_mask(
    seq_len: int,
    sliding_window: int | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    attention_mask = torch.ones(1, seq_len, seq_len, dtype=torch.bool, device=device)
    if sliding_window:
        attention_mask = attention_mask.tril(sliding_window).triu(-sliding_window)
    return attention_mask


@auto_docstring
class UltrabertModel(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(
            config.vocab_size, config.hidden_size, self.padding_idx
        )

        self.layers = nn.ModuleList(
            [UltrabertLayer(config, i) for i in range(config.num_hidden_layers)]
        )
        self.norm = UltrabertRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = nn.ModuleDict()
        self.gradient_checkpointing = False
        for base in set(config.rope_theta):
            name = str(base).replace(".", "_")
            self.rotary_emb[name] = RotaryEmbedding(base, config.head_dim)

        # Initialize weights and apply final processing
        self.post_init()

    @merge_with_config_defaults
    @capture_outputs
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutput:
        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("must specify exactly one of input_ids or inputs_embeds")
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        if position_ids is None:
            position_ids = torch.arange(inputs_embeds.size(1))
            position_ids = position_ids.to(inputs_embeds.device).unsqueeze(0)
        if (
            attention_mask is None
            and self.config._attn_implementation != "flash_attention_2"
        ):
            attention_mask = inputs_embeds.new_ones(inputs_embeds.shape[:2])
        varlen_kwargs = {
            k: kwargs[k]
            for k in ("cu_seq_lens_q", "cu_seq_lens_k", "max_length_q", "max_length_k")
            if kwargs.get(k) is not None
        }

        # Create attention masks for different local window sizes and store by window.
        attention_mask_dict = {}
        for local_window in list(set(self.config.attention_windows)) + [None]:
            if self.config._attn_implementation == "flash_attention_2":
                # For FlashAttention 2, rely on the `sliding_window` keyword instead of
                # constructing masks manually.
                attention_mask_dict[local_window] = attention_mask
            else:
                window_mask = create_local_attention_mask(
                    seq_len=attention_mask.size(1),
                    sliding_window=local_window,
                    device=attention_mask.device,
                )
                window_mask = window_mask & attention_mask.bool().unsqueeze(1)
                attention_mask_dict[local_window] = (
                    torch.where(window_mask, 0.0, torch.finfo(inputs_embeds.dtype).min)
                    .to(inputs_embeds.dtype)
                    .unsqueeze(1)
                )

        # Create position embeddings to be shared across the decoder layers.
        hidden_states, position_embeddings = inputs_embeds, {}
        for base in set(self.config.rope_theta):
            rotary_emb = self.rotary_emb[str(base).replace(".", "_")]
            position_embeddings[base] = rotary_emb(hidden_states, position_ids)

        for i, decoder_layer in enumerate(self.layers[: self.config.num_hidden_layers]):
            local_window = self.config.attention_windows[i]
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=attention_mask_dict[local_window],
                position_ids=position_ids,
                position_embeddings=position_embeddings[self.config.rope_theta[i]],
                sliding_window=(
                    local_window + 1
                    if local_window is not None
                    and self.config._attn_implementation == "flash_attention_2"
                    else local_window
                ),
                **varlen_kwargs,
            )
        hidden_states = self.norm(hidden_states)
        return BaseModelOutput(last_hidden_state=hidden_states)


class UltrabertPredictionHead(nn.Module):
    def __init__(self, config: UltrabertConfig):
        super().__init__()
        if (dim := config.prediction_head_size) is not None:
            self.linear1 = nn.Linear(config.hidden_size, dim, bias=False)
            self.linear2 = nn.Linear(dim, config.hidden_size, bias=False)
            self.act_fn = ACT2FN["gelu_pytorch_tanh"]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "linear1"):
            return self.act_fn(self.linear2(self.act_fn(self.linear1(x))))
        return x


@auto_docstring
class UltrabertForMaskedLM(UltrabertPreTrainedModel):
    _tied_weights_keys = {"cls.weight": "ultrabert.embed_tokens.weight"}

    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.ultrabert = UltrabertModel(config)
        self.head = UltrabertPredictionHead(config)
        self.cls = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Initialize weights and apply final processing
        self.post_init()

    def get_output_embeddings(self):
        return self.cls

    def set_output_embeddings(self, new_embeddings):
        self.cls = new_embeddings

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | MaskedLMOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should be in `[-100, 0, ...,
            config.vocab_size]` (see `input_ids` docstring) Tokens with indices set to `-100` are ignored (masked), the
            loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        prediction_scores = self.cls(self.head(outputs[0]))

        masked_lm_loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()  # -100 index = padding token
            masked_lm_loss = loss_fct(
                prediction_scores.view(-1, self.config.vocab_size), labels.view(-1)
            )

        return MaskedLMOutput(
            loss=masked_lm_loss,
            logits=prediction_scores,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring(
    custom_intro="""
    Ultrabert Model transformer with a sequence classification/regression head on top (a linear layer on top of the pooled
    output) e.g. for GLUE tasks.
    """
)
class UltrabertForSequenceClassification(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.ultrabert = UltrabertModel(config)
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(config.classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        logits = self.classifier(self.dropout(outputs[0][:, 0]))

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (
                    labels.dtype == torch.long or labels.dtype == torch.int
                ):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring
class UltrabertForQuestionAnswering(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels

        self.ultrabert = UltrabertModel(config)
        self.qa_outputs = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        start_positions: torch.Tensor | None = None,
        end_positions: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | QuestionAnsweringModelOutput:
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        logits = self.qa_outputs(outputs[0])
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()
        end_logits = end_logits.squeeze(-1).contiguous()

        total_loss = None
        if start_positions is not None and end_positions is not None:
            if len(start_positions.size()) > 1:
                start_positions = start_positions.squeeze(-1)
            if len(end_positions.size()) > 1:
                end_positions = end_positions.squeeze(-1)

            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)

            loss_fct = CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            total_loss = (start_loss + end_loss) / 2

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring
class UltrabertForTokenClassification(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.ultrabert = UltrabertModel(config)
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(config.classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | TokenClassifierOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the token classification loss. Indices should be in `[0, ..., config.num_labels - 1]`.
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

        sequence_output = outputs[0]

        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/atlas_worker.py
"""Optional likelihood expert; core predictions already exist before this worker."""
import os,sys,json,time,gc,argparse
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from stage_assets import staged_legacy_asset
from reference_helpers import LABELS

p=argparse.ArgumentParser()
p.add_argument('--query',required=True)
p.add_argument('--out',required=True)
a=p.parse_args()
start=time.monotonic()
query=pd.read_csv(a.query,dtype={'id':str});query['text']=query.text.fillna('').astype(str)
sys.path.insert(0,str(Path(__file__).resolve().parent/'inline_support'))
from ntl_atlas.model import load_model
from ntl_rlcd.data import LABELS as ATLAS_LABELS
from safetensors.torch import load_file
def score_staged(base, bundle):
    cfg=json.loads((bundle/'atlas_bundle.json').read_text())['training_config']
    model,tokenizer=load_model(str(base),ATLAS_LABELS,cfg,device='auto')
    model.load_trainable(load_file(str(bundle/'adapters.safetensors')));model.eval()
    tokens=tokenizer(query.text.tolist(),truncation=True,max_length=cfg['max_length'])['input_ids']
    # There is no next-token likelihood for a passage of fewer than two tokens.
    valid=np.array([i for i,t in enumerate(tokens) if len(t)>=2],dtype=int)
    order=valid[np.argsort([len(tokens[i]) for i in valid],kind='stable')]
    out=np.full((len(query),18),1/18,dtype=np.float32)
    with torch.inference_mode():
        for b in range(0,len(order),4):
            ix=order[b:b+4]
            batch=tokenizer.pad({'input_ids':[tokens[i] for i in ix]},padding=True,return_tensors='pt').to(model.device)
            probabilities,_=model.score(batch['input_ids'],batch['attention_mask'],source_chunk=3)
            out[ix]=probabilities.float().cpu().numpy()
            del probabilities,batch
            if b%100==0:print(json.dumps(dict(stage='ATLAS_QUERY',done=b+len(ix),total=len(order),seconds=time.monotonic()-start)),flush=True)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return out

with staged_legacy_asset('atlas_manifest', parent=True) as bundle:
    with staged_legacy_asset('qwen3_8b_base') as staged_base:
        out = score_staged(staged_base, bundle)
columns=[ATLAS_LABELS.index(l) for l in LABELS]
out=out[:,columns]
assert np.isfinite(out).all()
np.testing.assert_allclose(out.sum(1),1,atol=1e-4)
np.savez_compressed(a.out,ids=query.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),probabilities=out)
print(json.dumps(dict(stage='ATLAS_COMPLETE',rows=len(query),seconds=time.monotonic()-start)),flush=True)


## Full-text GSQ3 inference


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/gsq_infer_model.py
"""Full64-layer GSQ inference on one T4; exact base weights, FP16 compute."""
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        # A bounded one-tensor CPU copy avoids writable-buffer warnings and
        # ensures no mutation can alter the read-only safetensors mapping.
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            # BF16 -> FP32 is exact: shift the original 16 bits into the high
            # half of IEEE FP32 on CPU. T4 never sees a BF16 conversion.
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        # Row selection occurs BEFORE unpacking. Never materialize all248320
        # decoded embedding rows or move the full packed table to GPU.
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                # The causal_conv1d extension is absent. The native fallback
                # retains identical causal convolution/SiLU operations.
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/gsq_infer_layers.py
"""Authentic GSQ packed weights with bounded FP16 T4 inference.

The3bit Triton decoding is the same operator that passed the authentic k_proj
BF16 forward/dX/LoRA proof. No re-quantization or full-model decompression.
"""
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    """Original ignored BF16 in_proj_a/b weights; not re-quantized."""
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/delta_turing.py
# Copyright 2025 The Qwen Team and The HuggingFace Inc. team. All rights reserved.
# Modifications copyright 2026 Vladislav Lavrentev.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy at http://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software distributed
# under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR
# CONDITIONS OF ANY KIND, either express or implied. See the License for the
# specific language governing permissions and limitations under the License.
"""Qwen/Hugging Face DeltaNet reference with batched triangular solve on T4.

Only the strict-lower inverse recurrence is changed. The remainder of the
reference chunk algorithm is retained verbatim, including all chunk states.
"""
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    # reshape to chunks
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    # chunk decay
    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    # If A is the original strictly-lower triangular attn, the reference
    # recurrence computes B=A+A*B, hence I+B=(I-A)^(-1). A batched triangular
    # solve replaces only those63 Python row updates. Keep this solve FP32
    # even inside the T4 FP16 autocast context; no FP16 triangular solver.
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    # for each chunk
    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/adapter.py
"""Exact final GSQ adapter contract; no partial checkpoint fallback."""
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/infer.py
"""Two isolated T4 workers for full-text GSQ features and exact row reassembly."""
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue  # No contrast: downstream uses plain logits and mask=False.
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64), features=features,
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()


## Sequential stages and final mixture

Core and complementary probabilities mix at 85/15. Selected rows then receive 20% ATLAS. The own full-text GSQ head contributes 7.5% to every row after ATLAS. Adaptive balancing and hierarchical typography follow this mixture.

In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/stage_dataset_weights.py
"""Sequential, explicit-file Dataset staging for local random reads."""
import json
import shutil
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

NF4_FILES = {'LICENSE': 11544, 'README.md': 27108, 'chat_template.jinja': 8952, 'config.json': 4222, 'generation_config.json': 202, 'merges.txt': 3353259, 'model-00001-of-00007.safetensors': 2542796928, 'model-00002-of-00007.safetensors': 3967729339, 'model-00003-of-00007.safetensors': 3973498949, 'model-00004-of-00007.safetensors': 3992447326, 'model-00005-of-00007.safetensors': 4000148996, 'model-00006-of-00007.safetensors': 98537568, 'model-00007-of-00007.safetensors': 849400424, 'model.safetensors.index.json': 396308, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759}
ULTRA_FOLD_FILES = {0: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 1: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 2: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 3: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 4: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}}

@contextmanager
def staged_dataset_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    source = Path(source)
    input_root = Path(input_root)
    temp_root = Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-', '').replace('_', '').isalnum()
    assert files and all('/' not in name and '..' not in name and size > 0
                         for name, size in files.items())
    total = sum(files.values())
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2**30, ('Insufficient /tmp disk for explicit staging', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    start = time.monotonic()
    try:
        for name, size in files.items():
            target = destination / name
            with (source / name).open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size
            print(json.dumps({'stage': label, 'file': name, 'bytes': size,
                              'elapsed_seconds': round(time.monotonic() - start, 2)}), flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps({'stage': label, 'event': 'STAGING_CLEANED'}), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/ensemble.py
"""Sequential GPU stages and one final CPU blend; no model is loaded here."""
import argparse
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from scipy.special import softmax

from core_math import LABELS, topic_groups, finalize
from stage_dataset_weights import staged_dataset_tree


def run(root, logs, stage, script, arguments, gpus='', extra_env=None):
    env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
               TOKENIZERS_PARALLELISM='false', CUDA_VISIBLE_DEVICES=gpus)
    env.update(extra_env or {})
    with (logs / (stage + '.log')).open('w', encoding='utf-8', buffering=1) as stream:
        code = subprocess.call([sys.executable, '-u', str(root / script), *map(str, arguments)],
                               stdout=stream, stderr=subprocess.STDOUT, cwd=root, env=env)
    if code:
        raise RuntimeError(f'{stage} failed with exit code {code}; see its stage log')


def aligned(path, ids, field):
    with np.load(path, allow_pickle=False) as data:
        saved_ids, labels = data['ids'].astype(str).tolist(), data['labels'].astype(str).tolist()
        assert len(saved_ids) == len(set(saved_ids)) == len(ids) and set(saved_ids) == set(ids)
        assert len(labels) == len(set(labels)) == 18 and set(labels) == set(LABELS)
        index = {v: i for i, v in enumerate(saved_ids)}
        value = data[field][[index[i] for i in ids]][:, [labels.index(l) for l in LABELS]]
    assert value.shape == (len(ids), 18) and np.isfinite(value).all()
    return value


def blend_all_rows(core, complementary, atlas, atlas_rows, gsq, gsq_share):
    """The three probability mixtures occur before any class balancing."""
    prior = .85 * softmax(core, axis=1) + .15 * softmax(complementary, axis=1)
    prior /= prior.sum(1, keepdims=True)
    prior[atlas_rows] = .8 * prior[atlas_rows] + .2 * atlas
    prior /= prior.sum(1, keepdims=True)
    mixed = (1.0 - gsq_share) * prior + gsq_share * gsq
    assert np.isfinite(mixed).all() and np.allclose(mixed.sum(1), 1)
    return mixed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=Path, required=True)
    parser.add_argument('--out', type=Path, required=True)
    parser.add_argument('--work', type=Path, default=Path('/kaggle/temp/ntl-gsq-v27-work'))
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    assert config['RELEASE_READY'] is True and all(config['gates'].values()), 'Release binding gate is closed'
    assert config['complementary_share'] == .15, 'Fixed complementary share differs'
    assert config['gsq_share'] == 0.075, 'Unexpected GSQ share'
    assert config['scored_reference_script_version'] == 363008892
    assert config['token_attention_q35_enabled'] is False
    assert all(spec['name'] != 'modernbert' for spec in config['models'])
    root = Path(__file__).resolve().parent
    work = args.work
    work.mkdir(parents=True, exist_ok=True)
    logs = Path('/kaggle/working/inference-logs')
    logs.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(config['paths']['test_csv'], dtype={'id': str})
    assert frame.columns.tolist() == ['id', 'text'] and len(frame) > 0 and frame.id.is_unique
    frame['text'] = frame.text.fillna('').astype(str)
    ids = frame.id.tolist()
    query = work / 'query.csv'
    frame.to_csv(query, index=False)
    groups = topic_groups(frame.text.tolist())
    group_path = work / 'Gte.npy'
    np.save(group_path, groups, allow_pickle=False)
    features = work / 'core-features'
    features.mkdir(exist_ok=True)
    # Each child exits and releases its complete GPU context before the next stage.
    for i, spec in enumerate(config['models']):
        run(root, logs, 'core-' + spec['name'], 'core_worker.py',
            ['--config', args.config, '--query', query, '--member', i, '--out', features / spec['name']], '0,1')
    run(root, logs, 'core-readouts-cpu', 'core_readout.py',
        ['--config', args.config, '--query', query, '--groups', group_path,
         '--features', features, '--out', work / 'core.npz'])
    core = aligned(work / 'core.npz', ids, 'logits')
    run(root, logs, 'complementary', 'legacy/comp_cached.py', [], '0,1', {
        'NTL_QUERY': str(query), 'NTL_PREFIT_ROOT': config['paths']['legacy_prefit'],
        'NTL_STYLE_ROOT': config['paths']['complement_style'],
        'NTL660_PREFIT_ROOT': config['paths']['ntl660_prefit'],
        'NTL_COMP_OUT': str(work / 'complementary.npy'),
        'PYTHONPATH': str(root),
    })
    comp_meta = json.loads((work / 'complementary.npy.json').read_text())
    assert comp_meta['ids'] == ids and comp_meta['labels'] == LABELS
    comp = np.load(work / 'complementary.npy', allow_pickle=False)
    assert comp.shape == core.shape and np.isfinite(comp).all()
    prior = .85 * softmax(core, axis=1) + .15 * softmax(comp, axis=1)
    prior /= prior.sum(1, keepdims=True)
    order = np.sort(prior, axis=1)
    atlas_rows = np.flatnonzero(order[:, -1] - order[:, -2] < .3) if len(ids) >= 50 else np.arange(len(ids))
    atlas = np.empty((0, 18))
    if len(atlas_rows):
        atlas_query = work / 'atlas-query.csv'
        frame.iloc[atlas_rows].to_csv(atlas_query, index=False)
        run(root, logs, 'atlas', 'legacy/atlas_worker.py', ['--query', atlas_query, '--out', work / 'atlas.npz'], '0,1')
        atlas = aligned(work / 'atlas.npz', [ids[i] for i in atlas_rows], 'probabilities')
        assert (atlas >= 0).all() and np.allclose(atlas.sum(1), 1, atol=1e-4)
    # Load new-transformers dependencies only in the own GSQ process tree.
    target = work / 'gsq-python'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check',
                    '--target', str(target), *config['paths']['qwen_wheels']], check=True)
    parquet = work / 'query.parquet'
    frame.to_parquet(parquet, index=False)
    paths = config['paths']
    # One sequential /tmp copy is shared read-only by the two isolated GPU workers.
    with staged_dataset_tree(paths['gsq_base'], config['gsq_base_inventory'], label='gsq3-base') as base:
        run(root, logs, 'gsq-two-t4', 'gsq/infer.py',
            ['--base', base, '--final', paths['gsq_final'], '--metadata', paths['gsq_metadata'],
             '--complete', paths['gsq_complete'], '--input', parquet, '--groups', group_path,
             '--output', work / 'gsq.npz', '--work-dir', work / 'gsq-workers', '--batch-size', 1],
            '', dict(PYTHONPATH=str(target)))
    with np.load(work / 'gsq.npz', allow_pickle=False) as own:
        assert np.array_equal(own['row_index'], np.arange(len(frame)))
        assert np.array_equal(own['group_code'], groups)
        labels = own['labels'].tolist()
        assert len(labels) == len(set(labels)) == 18 and set(labels) == set(LABELS)
        assert config['gsq_head'] == 'plain', 'A centered mixture needs separate evidence and an explicit policy'
        gsq = softmax(own['plain_logits'][:, [labels.index(l) for l in LABELS]], axis=1)
        assert np.isfinite(gsq).all() and np.allclose(gsq.sum(1), 1)
    mixed = blend_all_rows(core, comp, atlas, atlas_rows, gsq, config['gsq_share'])
    np.savez(work / 'pre-balance.npz', ids=np.asarray(ids), labels=np.asarray(LABELS),
             core=core, complementary=comp, atlas_rows=atlas_rows, atlas=atlas, gsq=gsq, probabilities=mixed)
    official = pd.read_csv(config['paths']['train_csv'], dtype={'id': str})
    assert len(official) == 2699 and official.id.is_unique
    official_texts = official.text.fillna('').astype(str).tolist()
    official_labels = official.model.map({label: i for i, label in enumerate(LABELS)})
    assert official_labels.notna().all()
    logits = finalize(mixed, frame.text.values, groups,
                      train_texts=official_texts, train_labels=official_labels.to_numpy(dtype=np.int64))
    np.savez(work / 'final-logits.npz', ids=np.asarray(ids), labels=np.asarray(LABELS), logits=logits)
    output = pd.DataFrame({'id': ids, 'model': np.asarray(LABELS)[logits.argmax(1)]})
    assert len(output) == len(frame) and output.id.is_unique and output.model.isin(LABELS).all()
    output.to_parquet(args.out, index=False)
    # Inspect the final on-disk writer, after all gates, balances and typography.
    saved = pd.read_parquet(args.out)
    assert saved.equals(output) and saved.id.tolist() == ids
    (logs / 'execution-contract.json').write_text(json.dumps({
        'mode': config['execution_mode'], 'rows': len(frame),
        'stages': [spec['name'] for spec in config['models']] + ['core-readouts-cpu', 'complementary', 'atlas', 'gsq-two-t4'],
        'gsq_share': config['gsq_share'], 'complementary_share': .15,
        'q35_token_attention_enabled': False, 'deberta_token_attention_enabled': True,
        'sinkhorn_strength': .45, 'sinkhorn_target_mix': .35,
        'typography_markers': 37, 'final_writer': 'ensemble.py:output.to_parquet',
    }, indent=2))


if __name__ == '__main__':
    main()


## Input preparation and execution

Each core model's base, adapter, head and metadata files are copied from explicit immutable inventories to temporary storage and removed after feature extraction releases the model. Hidden-state tensors are released after pooling each batch and before an overflow retry, so one batch's states do not overlap the next forward or adapter update. The six V5 checkpoints retain their original order. Every GPU child exits before the next stage.

The complementary GPU child and its fitted readouts are active at 15% mixture share. Each complementary backbone is copied sequentially to temporary storage from its fixed file inventory. A LoRA trunk remains staged across its adapters, while each adapter and tokenizer directory is staged and released in the original order. The five family gates update complementary logits before they are saved. DeBERTa token-attention is active and stages its backbone and head; Qwen3.5 token-attention is disabled. ATLAS then stages its own base and adapter bundle. Models are released before temporary files are removed.

The GSQ base is then copied once to temporary storage, shared read-only by two isolated T4 workers and removed after both workers finish. GSQ retains every passage and its complete token sequence.

Final probabilities receive adaptive marginal balancing and the 37-marker hierarchical typography correction. The last writer saves the corrected per-passage argmax and verifies the saved IDs, row order and label schema.


In [ ]:
import shutil
ULTRA_ROOT = Path('/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl')
ULTRA_FILES = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
for fold in range(5):
    original_fold = ULTRA_ROOT / f'fold{fold}'
    local_fold = RUNTIME / 'legacy/inline_support/ultra' / f'fold{fold}'
    local_fold.mkdir(parents=True, exist_ok=True)
    for name in ULTRA_FILES:
        original_file = original_fold / name
        assert original_file.is_file(), original_file
        local_file = local_fold / name
        if local_file.is_symlink():
            assert local_file.resolve() == original_file.resolve(), local_file
        else:
            assert not local_file.exists(), local_file
            local_file.symlink_to(original_file)
    for name in ['configuration_ultrabert.py', 'modeling_ultrabert.py']:
        shutil.copyfile(RUNTIME / 'legacy/inline_support/ultra' / name, local_fold / name)


In [ ]:
NUMERIC_CACHE_PARTS = {'input_root': '/kaggle/input/ntl-olmo2-native24-final-inference-assets', 'parts': [{'name': 'ntl660-prefit.part000.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part001.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part002.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part003.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part004.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part005.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part006.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part007.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part008.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part009.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part010.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part011.bin', 'bytes': 963274}], 'archive_bytes': 277787338, 'cache_path': '/kaggle/temp/ntl660-prefit'}
CACHE_LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
if not preview:
    import shutil
    import stat
    import zipfile
    from pathlib import PurePosixPath
    parts_root = Path(NUMERIC_CACHE_PARTS['input_root'])
    cache_target = Path(NUMERIC_CACHE_PARTS['cache_path'])
    assembled_archive = Path('/kaggle/temp/ntl660-prefit.zip')
    assert cache_target == Path('/kaggle/temp/ntl660-prefit')
    assert CONFIG['paths']['ntl660_prefit'] == str(cache_target)
    assert not cache_target.exists(), 'Refusing to overwrite a prior materialized numeric cache'
    assert [part['name'] for part in NUMERIC_CACHE_PARTS['parts']] == [f'ntl660-prefit.part{i:03d}.bin' for i in range(12)]
    assembled_bytes = 0
    created_cache = False
    try:
        with assembled_archive.open('wb') as joined:
            for part in NUMERIC_CACHE_PARTS['parts']:
                source_part = parts_root / part['name']
                assert source_part.is_file() and source_part.stat().st_size == part['bytes']
                copied_bytes = 0
                with source_part.open('rb') as source:
                    while block := source.read(16 * 1024 * 1024):
                        joined.write(block)
                        copied_bytes += len(block)
                assert copied_bytes == part['bytes']
                assembled_bytes += copied_bytes
        assert assembled_bytes == assembled_archive.stat().st_size == NUMERIC_CACHE_PARTS['archive_bytes'] == 277787338
        with zipfile.ZipFile(assembled_archive) as archive:
            entries = archive.infolist()
            assert len(entries) == 51 and len({entry.filename for entry in entries}) == 51
            for entry in entries:
                relative = PurePosixPath(entry.filename)
                assert entry.filename and '\\' not in entry.filename and not relative.is_absolute()
                assert '..' not in relative.parts and '.' not in relative.parts
                assert relative.as_posix() == entry.filename and not entry.is_dir()
                assert not stat.S_ISLNK(entry.external_attr >> 16)
                assert entry.filename == 'COMPLETE' or relative.suffix in {'.json', '.npz', '.joblib', '.md'}
            cache_manifest = json.loads(archive.read('manifest.json'))
            assert cache_manifest['status'] == 'COMPLETE'
            assert cache_manifest['source']['script_version_id'] == 363008892
            assert cache_manifest['labels'] == CACHE_LABELS
            assert cache_manifest['core']['svc4']['rows'] == 123828
            assert cache_manifest['core']['matched_clean_rows_after_dedup'] == 24443
            assert len(cache_manifest['artifacts']) == 48
            declared_files = {item['path']: item['bytes'] for item in cache_manifest['artifacts']}
            assert len(declared_files) == 48
            assert set(declared_files) | {'manifest.json', 'COMPLETE', 'README.md'} == {entry.filename for entry in entries}
            for entry in entries:
                if entry.filename in declared_files:
                    assert entry.file_size == declared_files[entry.filename]
            assert archive.read('COMPLETE') == b'ntl660-numeric-v1\n'
            cache_target.mkdir(parents=True, exist_ok=False)
            created_cache = True
            archive.extractall(cache_target)
        assert (cache_target/'COMPLETE').is_file()
        for name, size in declared_files.items():
            assert (cache_target/name).is_file() and (cache_target/name).stat().st_size == size
        print(json.dumps({'mode': 'lossless_numeric_cache_parts', 'parts': 12, 'files': 51,
                          'archive_bytes': assembled_bytes, 'manifest_status': 'COMPLETE',
                          'source_script_version_id': 363008892, 'runtime_source_in_parts': False}), flush=True)
    except Exception:
        if created_cache:
            assert cache_target == Path('/kaggle/temp/ntl660-prefit')
            shutil.rmtree(cache_target)
        raise
    finally:
        assembled_archive.unlink(missing_ok=True)


In [ ]:
if not preview:
    assert CONFIG['RELEASE_READY'] is True and all(CONFIG['gates'].values()), 'Release binding gate is closed'
    import subprocess
    subprocess.run([sys.executable,'-u',str(RUNTIME/'ensemble.py'),'--config',str(RUNTIME/'config.json'),
                    '--out',str(OUTPUT)],check=True)
    actual = pd.read_csv(DATA,dtype={'id':str})
    result = pd.read_parquet(OUTPUT)
    assert result.columns.tolist() == ['id','model'] and result.id.tolist() == actual.id.tolist()
    assert result.model.notna().all()
display(pd.read_parquet(OUTPUT).head())
